# Playground Series S6E9 / 「EV Adoption Tokens: XGBoost + TinyTokenTransformer」解説付き写し

- **コンペ**: [Predicting Electric Vehicle Purchases (Playground Series - Season 6 Episode 9)](https://www.kaggle.com/competitions/playground-series-s6e9) (2026年9月1日開始 / 9月30日終了、255チーム)
- **原著者**: CHOVY (chovyxu)
- **元notebook**: https://www.kaggle.com/code/chovyxu/ev-adoption-tokens-xgboost-tinytokentransformer
- **Best Score**: 0.94539 (Version 2) / 17 votes・Bronze
- **ライセンス**: Apache 2.0

> ⚠️ これは**学習目的の解説付き写し**です。原著のコードは変更しておらず、出力は含んでいません。各コードセルの直前に日本語の解説Markdownセルを挿入しています。

---

## タスクと評価指標

**タスク**: 個人属性(年齢、年収、通勤距離、充電設備の有無、環境意識、補助金の有無、都市タイプ、現在の車種など)から、**その人が電気自動車(EV)を買うか(`Will_Buy_EV`, Yes/No)**の確率を予測する二値分類。データは実データから生成された**合成データ(synthetic data)**です。

**評価指標**: **ROC AUC**(Area Under the Receiver Operating Characteristic Curve)。

### ROC AUC とは何か

予測確率でサンプルを降順に並べたとき、**「無作為に選んだ正例のスコア > 無作為に選んだ負例のスコア」となる確率**です。等価に、ROC曲線(横軸=偽陽性率、縦軸=真陽性率)の下側面積。

- 0.5 = でたらめ、1.0 = 完璧。
- **重要な性質: 単調変換で不変**。予測値を `p` から `log(p/(1-p))` にしても、順位が変わらなければ AUC は1ミリも動きません。**AUC は順位だけを見る指標**です。

### なぜこのタスクに AUC なのか

- **正例率が偏っていても意味が保たれる**。Accuracy はしきい値0.5に依存し、正例が10%なら「全員No」で90%になってしまいます。AUC はしきい値に依存しないので、不均衡でも比較可能です。
- **マーケティングの実務と合う**。「EVを買いそうな順に並べて上位N人に販促する」という使い方では、確率の絶対値ではなく**順位**が意味を持ちます。AUC はまさにそれを測ります。
- **Playground の合成データ向き**。合成データは確率の絶対水準(キャリブレーション)が実世界とずれがちなので、順位で評価する方が公平です。

### この notebook の設計が AUC をどう狙っているか

- **最終ブレンドの選択基準を AUC にし、logloss をタイブレークに使う**(セル12)。AUC が同点のときだけ logloss で選ぶので、**順位性能を最優先しつつ、確率としても壊れていないものを選ぶ**という二段構えです。
- **ブレンド候補に `prob` / `rank01` / `logit_rank01` の3つの「見方」を用意**。AUC は順位不変なので、**確率をそのまま混ぜるより順位に直してから混ぜた方が、スケールの違うモデル同士を公平に足せます**。ここは AUC 特有の設計です。
- **リーク対策を fold の中に閉じ込める**。target encoding を fold 外で作ると OOF AUC が実力以上に出て、LBで落ちます。この notebook は target-free な特徴(全データで作ってよいもの)と target-mean な特徴(fold内でしか作ってはいけないもの)を**クラスレベルで分離**しています。

---

## この notebook の中心アイデア:「表の1行を小さな言語に翻訳する」

タイトルの *tokens* が示す通り、この notebook は**表形式データの1行を「トークン列」として扱います**。

```
        1行のデータ (Age=34, Income=52000, City_Type="Urban", ...)
                          │
        ┌─────────────────┴─────────────────┐
        │                                   │
  [数値フィールドトークン]            [カテゴリフィールドトークン]
  1フィールド = 1トークン             1フィールド = 1トークン
  各トークンの中身:                   各トークンの中身:
   ・ロバスト標準化した値              ・カテゴリID(スケール済)
   ・欠損フラグ                        ・出現頻度 / log頻度
   ・2乗・絶対値                       ・レアかどうか
   ・sin/cos フーリエ基底(f=1,2,4,8)  ・フィールドの濃度
   ・局所ガウス基底                    ・one-hot 4スロット
   ・分位ビン頻度・丸め値頻度
        │                                   │
        └─────────────────┬─────────────────┘
                          │
        ┌─────────────────┴──────────────────┐
        │                                    │
  [横に全部つなげた2D行列]            [トークン列のまま]
        ↓                                    ↓
   XGBoost                          TinyTokenTransformer
   (分割で非線形を拾う)              ([CLS]トークン + self-attention)
        │                                    │
        └────────→ OOF貪欲ブレンド ←─────────┘
                          ↓
                   submission.csv
```

**2つのモデルに「同じ情報の別の見せ方」を渡す**のがポイントです。XGBoost は木の分割で特徴を使うので**幅の広い平らな行列**が向き、Transformer は attention でフィールド間の関係を学ぶので**フィールド=トークンの並び**が向きます。同じ特徴量から2つのビューを作り、最後に混ぜることで**アーキテクチャの多様性**を稼ぎます。

> 💡 用語: **OOF (Out-Of-Fold) 予測** = k分割交差検証で、各サンプルを「それを学習に使っていないモデル」が予測した値。学習データ全体分の「未見データに対する予測」が手に入るので、アンサンブルの重み探索に使えます。


# XGBoost + TinyTokenTransformer for EV Purchase Classification

The target is binary (`Will_Buy_EV`), so every model produces a probability for the positive class:

| Model | Input view | Strength |
|:--|:--|:--|
| XGBoost | flat numeric + categorical token matrix | split-friendly nonlinear tabular baseline |
| TinyTokenTransformer | field-token sequence plus dense summary tokens | attention over numeric and categorical fields |

The theme stays the same: translate a tabular row into a small token language, train two different model trunks, then blend with leakage-safe OOF predictions.


## Design Map

1. Load S6E9 data from Kaggle input folders or the local zip.
2. Encode `Yes/No` target labels as positive-class probabilities.
3. Run compact EDA for numeric and categorical token evidence.
4. Build target-free tokens:
   - robust numeric scaling
   - Fourier and local numeric basis tokens
   - frequency tokens for exact, rounded, and category values
   - categorical field tokens and one-hot blocks
   - numeric-numeric and numeric-category interactions
   - PCA/SVD row-geometry tokens
5. Add leakage-safe target-mean tokens inside each CV fold.
6. Train XGBoost and TinyTokenTransformer.
7. Search OOF blend weights and write `submission.csv`.


### セル1: インポートとグローバル設定

**何をしているか**
標準ライブラリと numpy/pandas を読み、matplotlib と IPython の `display` は `try/except` で「無ければ無いなりに動く」ようにフォールバックします。そのあと**すべてのハイパーパラメータを大文字定数として1か所に集約**します。

主な定数: `SEED=2026`、`N_SPLITS=5`(CV分割数)、`N_BINS=20`(数値の分位ビン数)、`PCA_COMPONENTS=10`、`TARGET_SMOOTHING=500.0` / `PAIR_TARGET_SMOOTHING=1500.0`(target encoding の平滑化)、`CAT_RARE_FREQUENCY=0.005`(これ未満の頻度のカテゴリはレア扱い)、`MAX_ONE_HOT_CARDINALITY=64`、`XGB_N_ESTIMATORS=20000`、`XGB_LEARNING_RATE=0.01`、`XGB_EARLY_STOPPING_ROUNDS=200`。実行スイッチ `RUN_XGBOOST` / `RUN_TINY_TOKEN_TRANSFORMER` も置かれています。

**なぜそうするのか**
- **なぜ定数を先頭に集めるのか**: 実験のたびに書き換える場所が1か所に固まるので、「どこを変えたか」が diff で一目で分かります。数百行下のマジックナンバーを探し回るのは事故のもとです。
- **なぜ木の本数を20000にして学習率を0.01にするのか**: GBDT では **`learning_rate` を小さくして本数を増やすほど汎化しやすい**という経験則があります。ただし本数を固定すると過学習するので、**early stopping(検証スコアが200ラウンド改善しなければ打ち切り)とセット**で使います。つまり20000は「上限」であって実際に使う本数ではありません。この組み合わせが GBDT のチューニングの基本形です。
- **なぜ `display` にフォールバックを置くのか**: Kaggle以外(ローカルのpython、CI)で回したときに `NameError` で止まらないため。**notebookは"実行環境が変わっても壊れないこと"に少し投資するだけで再利用性が跳ね上がります**。


In [ ]:
from __future__ import annotations

import gc
import importlib.util
import json
import math
import random
import time
import warnings
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    MPL_AVAILABLE = True
except Exception:
    MPL_AVAILABLE = False

try:
    display
except NameError:
    try:
        from IPython.display import display
    except Exception:
        display = print

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)

ID_COL = "id"
MISSING_TOKEN = "__MISSING__"
UNKNOWN_TOKEN = "__UNKNOWN__"
EPS = 1e-7
SEED = 2026

RUN_XGBOOST = True
RUN_XGBOOST_VARIANTS = True
RUN_TINY_TOKEN_TRANSFORMER = True

N_SPLITS = 5
N_BINS = 20
PCA_COMPONENTS = 10
PCA_SAMPLE_N = 80_000
TARGET_SMOOTHING = 500.0
PAIR_TARGET_SMOOTHING = 1_500.0
CAT_RARE_FREQUENCY = 0.005
MAX_ONE_HOT_CARDINALITY = 64

XGB_N_ESTIMATORS = 20000
XGB_LEARNING_RATE = 0.01
XGB_EARLY_STOPPING_ROUNDS = 200
XGB_USE_SCALE_POS_WEIGHT = False

TTT_MAX_EPOCHS = 20
TTT_PATIENCE = 4
TTT_BATCH_SIZE = 4096
TTT_LR = 2e-3
TTT_D_MODEL = 96
TTT_N_LAYERS = 2
TTT_N_HEADS = 4
TTT_DROPOUT = 0.10
TTT_USE_POS_WEIGHT = False

BLEND_GRID_STEP = 0.01

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path("outputs/s6e9_xgb_tinytransformer_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

XGBOOST_AVAILABLE = importlib.util.find_spec("xgboost") is not None
TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

print("Output directory:", OUTPUT_DIR)
print("xgboost available:", XGBOOST_AVAILABLE)
print("torch available:", TORCH_AVAILABLE)


## 1. Data Loading

The loader first searches common Kaggle input paths, then falls back to local zip names such as `playground-series-s6e9.zip` or `s6e9.zip`.


### セル2: データ読み込み(パスの多段フォールバック)

**何をしているか**
`find_data_source()` が、`train.csv` / `test.csv` / `sample_submission.csv` が3つそろっている場所を順に探します。

1. `/kaggle/input/playground-series-s6e9`
2. `/kaggle/input/competitions/playground-series-s6e9`
3. カレントディレクトリ
4. `playground-series-s6e9.zip` / `s6e9.zip` などのzip
5. それでも見つからなければ `/kaggle/input/**/train.csv` を再帰探索

そのあと目的変数 `Will_Buy_EV` の Yes/No を 1/0 に変換し、数値列とカテゴリ列を仕分けます。

**なぜそうするのか**
- **なぜここまでしつこく探すのか**: Playground の notebook は「Copy & Edit されて別環境で回される」ことが前提です。マウント位置は UI 追加か API 追加かで変わり、ローカルではzipのまま置かれることもあります。**このセルが通れば以降は環境差を気にしなくてよい**、という防波堤を1か所に作っています。
- **なぜ Yes/No を明示的にマッピングするのか**: `pd.factorize` や `astype('category').cat.codes` に任せると、**アルファベット順や出現順で 0/1 が決まる**ため、train と test で反転する可能性があります。二値分類で正例と負例が入れ替わると AUC は `1 - AUC` になり、0.945 が 0.055 になります。**どちらを正例とするかは必ず明示する**のが鉄則です。
- **なぜ数値/カテゴリを最初に仕分けるのか**: 以降のトークン生成が、この2つのリスト(`numeric_cols` / `categorical_cols`)を軸に完全に分岐するからです。ここでの仕分けが特徴量設計の骨格になります。


In [ ]:
def read_csv_from_zip(zip_path: Path, member: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(member) as handle:
            return pd.read_csv(handle)


def find_data_source() -> Path:
    directory_candidates = [
        Path("/kaggle/input/playground-series-s6e9"),
        Path("/kaggle/input/competitions/playground-series-s6e9"),
        Path("."),
    ]
    for path in directory_candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists() and (path / "sample_submission.csv").exists():
            return path

    zip_candidates = [
        Path("playground-series-s6e9.zip"),
        Path("s6e9.zip"),
        Path("/kaggle/input/playground-series-s6e9.zip"),
        Path("/kaggle/input/s6e9.zip"),
    ]
    for path in zip_candidates:
        if path.exists():
            return path

    if Path("/kaggle/input").exists():
        for train_path in Path("/kaggle/input").glob("**/train.csv"):
            folder = train_path.parent
            if (folder / "test.csv").exists() and (folder / "sample_submission.csv").exists():
                return folder
        for zip_path in Path("/kaggle/input").glob("**/*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as zf:
                    names = set(zf.namelist())
                if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(names):
                    return zip_path
            except zipfile.BadZipFile:
                continue

    raise FileNotFoundError("Could not find train.csv, test.csv, and sample_submission.csv.")


def load_competition_data(source: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if source.is_dir():
        return (
            pd.read_csv(source / "train.csv"),
            pd.read_csv(source / "test.csv"),
            pd.read_csv(source / "sample_submission.csv"),
        )
    return (
        read_csv_from_zip(source, "train.csv"),
        read_csv_from_zip(source, "test.csv"),
        read_csv_from_zip(source, "sample_submission.csv"),
    )


def infer_target_column(train_frame: pd.DataFrame, sample_frame: pd.DataFrame) -> str:
    target_candidates = [c for c in sample_frame.columns if c != ID_COL]
    assert len(target_candidates) == 1, target_candidates
    target = target_candidates[0]
    assert target in train_frame.columns, f"{target} is not present in train.csv"
    return target


def encode_binary_target(series: pd.Series) -> tuple[np.ndarray, str, dict[object, int]]:
    if pd.api.types.is_numeric_dtype(series):
        values = pd.to_numeric(series, errors="coerce")
        unique_values = sorted(values.dropna().unique().tolist())
        if not set(unique_values).issubset({0, 1, 0.0, 1.0}):
            raise ValueError(f"Expected binary numeric target, got {unique_values[:10]}")
        return values.to_numpy(dtype="float32"), "1", {0: 0, 1: 1}

    normalized = series.astype(str).str.strip()
    lowered = normalized.str.lower()
    positive_values = {"yes", "y", "true", "t", "1", "positive"}
    negative_values = {"no", "n", "false", "f", "0", "negative"}
    unknown = sorted(set(lowered.unique()) - positive_values - negative_values)
    if unknown:
        raise ValueError(f"Unexpected binary target labels: {unknown}")
    y_values = lowered.isin(positive_values).astype("float32").to_numpy()
    positive_label = normalized[lowered.isin(positive_values)].iloc[0] if y_values.sum() > 0 else "1"
    mapping = {label: int(label.lower() in positive_values) for label in sorted(normalized.unique())}
    return y_values, str(positive_label), mapping


DATA_SOURCE = find_data_source()
train, test, sample_submission = load_competition_data(DATA_SOURCE)

TARGET = infer_target_column(train, sample_submission)
y, POSITIVE_LABEL, TARGET_MAPPING = encode_binary_target(train[TARGET])

feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET]]
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]
target_prior = float(y.mean())

print("Data source:", DATA_SOURCE)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Target:", TARGET)
print("Target mapping:", TARGET_MAPPING)
print("Positive label:", POSITIVE_LABEL)
print("Positive rate:", f"{target_prior:.6f}")
print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)
display(train.head())


## 2. EDA as Token Evidence

S6E9 mixes continuous usage/cost signals with low-cardinality categorical context. The token layer therefore needs both smooth numeric basis functions and category/frequency/interaction tokens.


### セル3: EDA ― トークン設計のための「証拠集め」

**何をしているか**
可視化のためのEDAではなく、**どんなトークンを作るべきかを決めるための計測**です。

- `safe_corr` — 各数値列と目的変数の相関(NaNを落としてから計算、ユニーク値が1つなら NaN を返す)。
- `exact_overlap` — **test の値のうち、train にも出現する値の割合**。
- 各数値列を `pd.qcut(q=20)` で20分位に切り、**ビンごとの目的変数平均**(=応答曲線)を計算。
- 欠損率、train/test それぞれのユニーク値数。

**なぜそうするのか**
- **なぜ「ビンごとの目的変数平均」を見るのか**: 相関係数は**線形関係しか捉えません**。年収とEV購入率が「低所得では低く、中所得で急上昇し、高所得で頭打ち」というS字なら、相関は小さく出ても情報は豊富にあります。分位ビンごとの平均を見れば**非線形な応答の形がそのまま見えます**。この形を見て「フーリエ基底や局所ガウス基底で近似しよう」という後段の設計につながります。
- **なぜ `exact_overlap` を測るのか(重要)**: これは合成データのコンペ特有の観点です。もし test に現れる値がほぼすべて train にも現れるなら、その列は実質**カテゴリのように扱えて、頻度エンコーディングや target encoding が強く効きます**。逆に overlap が低ければ「連続量として滑らかに扱うべき」です。**「その列を連続として扱うか離散として扱うか」を、勘ではなくデータで決めている**わけです。
- **なぜ欠損率を見るのか**: 欠損の入り方には情報があることが多く(合成データでは特に)、後段で `missing` フラグを**明示的なトークン成分として持たせる**根拠になります。


In [ ]:
def safe_corr(x: pd.Series, y_values: np.ndarray) -> float:
    numeric = pd.to_numeric(x, errors="coerce")
    frame = pd.DataFrame({"x": numeric, "y": y_values}).dropna()
    if len(frame) < 3 or frame["x"].nunique() <= 1:
        return np.nan
    return float(frame["x"].corr(frame["y"]))


def exact_overlap(train_series: pd.Series, test_series: pd.Series) -> float:
    train_values = set(train_series.dropna().astype(str).unique().tolist())
    test_values = set(test_series.dropna().astype(str).unique().tolist())
    return len(train_values & test_values) / max(1, len(test_values))


numeric_eda_rows = []
for col in numeric_cols:
    bins = pd.qcut(train[col], q=20, duplicates="drop")
    curve = pd.DataFrame({"bin": bins, "target": y}).groupby("bin", observed=True)["target"].mean().to_numpy(dtype="float32")
    numeric_eda_rows.append(
        {
            "column": col,
            "missing_rate": train[col].isna().mean(),
            "train_unique": train[col].nunique(dropna=False),
            "test_unique": test[col].nunique(dropna=False),
            "test_exact_overlap": exact_overlap(train[col], test[col]),
            "point_biserial_corr": safe_corr(train[col], y),
            "qbin_positive_rate_range": float(np.nanmax(curve) - np.nanmin(curve)),
        }
    )

categorical_eda_rows = []
for col in categorical_cols:
    rates = pd.DataFrame({"category": train[col].astype(str), "target": y}).groupby("category", observed=True)["target"].agg(["mean", "count"])
    weighted_rates = rates.loc[rates["count"] >= 100, "mean"] if len(rates) else pd.Series(dtype="float32")
    rate_range = float(weighted_rates.max() - weighted_rates.min()) if len(weighted_rates) else 0.0
    categorical_eda_rows.append(
        {
            "column": col,
            "missing_rate": train[col].isna().mean(),
            "train_unique": train[col].nunique(dropna=False),
            "test_unique": test[col].nunique(dropna=False),
            "test_exact_overlap": exact_overlap(train[col], test[col]),
            "category_positive_rate_range": rate_range,
            "largest_category_share": float(train[col].astype(str).value_counts(normalize=True).iloc[0]),
        }
    )

numeric_eda = pd.DataFrame(numeric_eda_rows).sort_values("qbin_positive_rate_range", ascending=False)
categorical_eda = pd.DataFrame(categorical_eda_rows).sort_values("category_positive_rate_range", ascending=False)

print("Numeric evidence")
display(numeric_eda)
print("Categorical evidence")
display(categorical_eda)

if MPL_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    axes[0].bar(["No", "Yes"], [1.0 - target_prior, target_prior], color=["#4C78A8", "#F58518"])
    axes[0].set_title("Target balance")
    axes[0].set_ylabel("share")

    if len(numeric_eda):
        plot_df = numeric_eda.sort_values("qbin_positive_rate_range")
        axes[1].barh(plot_df["column"], plot_df["qbin_positive_rate_range"], color="#54A24B")
        axes[1].set_title("Numeric qbin target movement")
        axes[1].set_xlabel("positive-rate range")

    if len(categorical_eda):
        plot_df = categorical_eda.sort_values("category_positive_rate_range")
        axes[2].barh(plot_df["column"], plot_df["category_positive_rate_range"], color="#E45756")
        axes[2].set_title("Categorical target movement")
        axes[2].set_xlabel("positive-rate range")

    plt.tight_layout()
    plt.show()


### セル4: train/test のドリフト可視化

**何をしているか**
数値列については **平均のズレを共通標準偏差で割った値** `|mean_train - mean_test| / pooled_std` と、標準偏差の比 `std_test / std_train` を計算して横棒グラフに。カテゴリ列については train と test の**頻度分布のL1距離**(各カテゴリの割合の差の絶対値の総和)を計算します。

**なぜそうするのか**
- **なぜドリフトを測るのか**: train と test で分布が違えば、train で最適化した特徴量やしきい値は test で効きません。これを **covariate shift(共変量シフト)** と呼びます。
- **標準偏差で割るのはなぜか**: 年収(単位:ドル、値は数万)と環境意識レベル(1〜5)では、生の平均差を比べても意味がありません。**共通の標準偏差で割ることで無次元化**し、列同士を比較できるようにしています(効果量の考え方と同じ)。
- **カテゴリにL1距離を使うのはなぜか**: 2つの離散分布の差を測る素直な尺度で、`0`(完全一致)から `2`(まったく重ならない)の範囲に収まり解釈しやすい。KLダイバージェンスと違い、片方が0の値があっても発散しません。
- **実務上の意味**: Playground の合成データは通常 train/test が同一の生成器から作られるのでドリフトは小さいはずです。**「小さいことを確認する」ことに価値があります**。もしここで大きなドリフトが出たら、その列は使い方を変えるか adversarial validation を検討する、という判断分岐になります。


In [ ]:
# Train/Test Feature Drift View

if MPL_AVAILABLE:
    rows = []
    for col in numeric_cols:
        tr = pd.to_numeric(train[col], errors="coerce")
        te = pd.to_numeric(test[col], errors="coerce")
        pooled_std = pd.concat([tr, te]).std()
        rows.append({
            "column": col,
            "mean_shift_std": abs(tr.mean() - te.mean()) / max(EPS, pooled_std),
            "std_ratio": te.std() / max(EPS, tr.std()),
        })

    drift = pd.DataFrame(rows).sort_values("mean_shift_std", ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    axes[0].barh(drift["column"], drift["mean_shift_std"], color="#4C78A8")
    axes[0].invert_yaxis()
    axes[0].set_title("Numeric Train/Test Mean Shift")
    axes[0].set_xlabel("absolute mean shift / pooled std")

    cat_rows = []
    for col in categorical_cols:
        tr_freq = train[col].astype(str).value_counts(normalize=True)
        te_freq = test[col].astype(str).value_counts(normalize=True)
        keys = sorted(set(tr_freq.index) | set(te_freq.index))
        l1 = sum(abs(tr_freq.get(k, 0.0) - te_freq.get(k, 0.0)) for k in keys)
        cat_rows.append({"column": col, "frequency_l1_shift": l1})

    cat_drift = pd.DataFrame(cat_rows).sort_values("frequency_l1_shift", ascending=False)
    axes[1].barh(cat_drift["column"], cat_drift["frequency_l1_shift"], color="#F58518")
    axes[1].invert_yaxis()
    axes[1].set_title("Categorical Train/Test Frequency Shift")
    axes[1].set_xlabel("L1 frequency distance")

    plt.tight_layout()
    plt.show()

    display(drift)
    display(cat_drift)

## 3. Metrics, Folds, and Utility Functions

The primary validation readout is AUC, with logloss and Brier score kept nearby so probability quality does not drift out of view.


### セル5: 評価関数と fold 生成(すべて自前実装)

**何をしているか**
`clip_prob`(確率を `[1e-7, 1-1e-7]` に丸める)、`sigmoid_np`、`binary_log_loss`、`brier_score`、そして **`roc_auc_score_np`(AUCのnumpy自前実装)**、層化fold生成 `make_stratified_folds` を定義します。

AUCの実装は「予測でソート → 同値をまとめて平均順位を振る → 正例の順位和から `n_pos*(n_pos+1)/2` を引く → `n_pos*n_neg` で割る」というランクベースの式です。

**なぜそうするのか**
- **なぜ sklearn を使わず自前実装なのか**: 依存を減らすためもありますが、教育的にはこちらが本質的です。**AUC が「正例の順位の平均」から直接計算できる**という事実が、コードを読むと腑に落ちます。これは Mann-Whitney U 統計量と AUC が等価であるという定理そのものです。だから AUC は順位不変であり、だからこそ後段の rank blending が正当化されます。
- **なぜ同値に平均順位を振るのか(`method='average'`)**: 予測が完全に同じ値のサンプルがあるとき、順序を勝手に決めると AUC が運で上下します。平均順位なら**同値の並び順に依存しない一意な値**になります。木モデルは同じ葉に落ちたサンプルに同一値を出すので、これは実際に起きます。
- **なぜ AUC 以外に logloss と Brier も持つのか**: AUC は順位しか見ないので、**予測確率がすべて0.999付近に張り付いていても AUC は満点になりえます**。それでは「確率」として使えません。logloss(対数損失)と Brier スコア(予測確率と実測の二乗誤差)を並べて監視することで、**順位を追いかけるうちに確率が壊れていないか**をチェックしています。
- **なぜ層化(stratified)なのか**: 各 fold の正例率を全体と揃えます。正例率が fold ごとにばらつくと、fold ごとの AUC の分散が大きくなり、**改善が誤差に埋もれて見えなくなります**。
- **なぜ `clip_prob` するのか**: logloss は `log(0) = -inf` で爆発します。両端を微小量でクリップするのは定番の安全策です。


In [ ]:
def clip_prob(pred: np.ndarray) -> np.ndarray:
    return np.clip(np.asarray(pred, dtype="float32"), EPS, 1.0 - EPS)


def sigmoid_np(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype="float32")
    return (1.0 / (1.0 + np.exp(-np.clip(logits, -40.0, 40.0)))).astype("float32")


def binary_log_loss(y_true: np.ndarray, pred: np.ndarray) -> float:
    pred = clip_prob(pred)
    y_true = np.asarray(y_true, dtype="float32")
    return float(-(y_true * np.log(pred) + (1.0 - y_true) * np.log(1.0 - pred)).mean())


def brier_score(y_true: np.ndarray, pred: np.ndarray) -> float:
    pred = clip_prob(pred)
    y_true = np.asarray(y_true, dtype="float32")
    return float(np.mean((pred - y_true) ** 2))


def roc_auc_score_np(y_true: np.ndarray, pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int8)
    pred = np.asarray(pred, dtype="float64")
    n_pos = int(y_true.sum())
    n_neg = int(len(y_true) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    order = np.argsort(pred, kind="mergesort")
    sorted_pred = pred[order]
    ranks = np.empty(len(pred), dtype="float64")
    start = 0
    while start < len(pred):
        stop = start + 1
        while stop < len(pred) and sorted_pred[stop] == sorted_pred[start]:
            stop += 1
        avg_rank = 0.5 * (start + 1 + stop)
        ranks[order[start:stop]] = avg_rank
        start = stop

    pos_rank_sum = ranks[y_true == 1].sum()
    auc = (pos_rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)


def make_stratified_folds(target: np.ndarray, n_splits: int, seed: int) -> list[tuple[np.ndarray, np.ndarray]]:
    rng = np.random.default_rng(seed)
    target = np.asarray(target, dtype=np.int8)
    fold_id = np.empty(len(target), dtype=np.int16)
    for label in [0, 1]:
        idx = np.flatnonzero(target == label)
        rng.shuffle(idx)
        fold_id[idx] = np.arange(len(idx)) % n_splits
    return [
        (np.flatnonzero(fold_id != fold), np.flatnonzero(fold_id == fold))
        for fold in range(n_splits)
    ]


def _safe_numeric_matrix(frame: pd.DataFrame, columns: list[str]) -> np.ndarray:
    if not columns:
        return np.zeros((len(frame), 0), dtype="float32")
    return frame[columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype="float32")


def _normalized_category_series(frame: pd.DataFrame, column: str) -> pd.Series:
    values = frame[column].astype("object")
    return values.where(values.notna(), MISSING_TOKEN).astype(str)


def _append(pieces: list[np.ndarray], names: list[str], values: np.ndarray, value_names: Iterable[str]) -> None:
    arr = np.asarray(values, dtype="float32")
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    pieces.append(arr)
    names.extend(list(value_names))


constant_pred = np.full(len(train), target_prior, dtype="float32")
print(f"Constant baseline AUC    : {roc_auc_score_np(y, constant_pred):.6f}")
print(f"Constant baseline logloss: {binary_log_loss(y, constant_pred):.6f}")
print(f"Constant baseline Brier  : {brier_score(y, constant_pred):.6f}")


## 4. Token State

`TokenFeatureState` stores train+test target-free statistics:

- robust numeric quantiles for scaling
- quantile-bin edges and frequencies
- rounded numeric-value frequencies
- category vocabularies and category frequencies
- PCA/SVD basis for global row geometry


### セル6: TokenFeatureState ― 「目的変数を使わない」統計量の保管庫

**何をしているか**
`@dataclass` として、**train と test の両方から計算してよい統計量**だけを保持します。

- ロバスト標準化のための分位点(`center` = 中央値、`low` / `high` = 例えば5%点と95%点)
- 各数値列の分位ビン境界 `qbin_edges` とビン頻度 `qbin_freq`
- 丸めた数値の出現頻度 `rounded_freq_maps`(小数第0位・1位・2位…)
- カテゴリ語彙 `category_values` / ID対応 `category_maps` / 頻度 `category_frequency_maps`
- 全行を使った **PCA/SVD の基底**(最大80,000行をサンプリングして学習)

`fit(train_frame, test_frame)` は train と test を**縦に連結してから**統計を取ります。

**なぜそうするのか**
- **なぜ train と test を混ぜてよいのか(ここが最重要)**: このクラスが持つのは**目的変数を一切使わない統計量**だけです。だから test の特徴量を混ぜても**ラベルのリークは起きません**。むしろ test の分布を反映した方が、頻度エンコーディングや標準化が test 側で安定します。これは transductive(トランスダクティブ)な前処理と呼ばれ、Kaggleでは標準的な手法です。
  - **逆に、目的変数を使う統計量(target encoding)は絶対にここに入れてはいけません**。だからこの notebook は次のセルで `TargetMeanEncoder` を**別クラスとして分離**しています。**「リークするもの」と「しないもの」をクラス境界で物理的に分ける**という設計判断が、この notebook で一番真似する価値のある構造です。
- **なぜ平均・標準偏差ではなく分位点でスケーリングするのか(ロバスト標準化)**: 年収のような裾の重い分布では、外れ値1つで平均も標準偏差も大きく動きます。中央値と分位範囲を使えば外れ値の影響を受けません。Transformer に入れる値は範囲が揃っている方が学習が安定します。
- **なぜ「丸めた値の頻度」を持つのか**: 合成データでは、生成器の都合で**特定の丸め値が不自然に多く出る**ことがあります(例:年収がちょうど50000の人が異常に多い)。この「不自然さ」自体が目的変数と相関することがあり、拾えば効きます。セル7bの forensic 特徴量はこの発想の拡張です。
- **なぜ PCA をサンプリングして学習するのか**: 全行で SVD を回すとメモリと時間を食います。8万行あれば主成分の推定は十分安定します。


In [ ]:
@dataclass
class TokenFeatureState:
    numeric_cols: list[str]
    categorical_cols: list[str]
    frequencies: tuple[int, ...] = (1, 2, 4, 8)
    local_centers: tuple[float, ...] = (-1.0, -0.5, 0.0, 0.5, 1.0)
    n_bins: int = N_BINS
    pca_components_count: int = PCA_COMPONENTS
    pca_sample_n: int = PCA_SAMPLE_N
    seed: int = SEED

    center: np.ndarray | None = None
    low: np.ndarray | None = None
    high: np.ndarray | None = None
    qbin_edges: dict[str, np.ndarray] | None = None
    qbin_freq: dict[str, np.ndarray] | None = None
    rounded_freq_maps: dict[tuple[str, int], dict[float, float]] | None = None
    category_values: dict[str, list[str]] | None = None
    category_maps: dict[str, dict[str, int]] | None = None
    category_frequency_maps: dict[str, dict[str, float]] | None = None
    all_row_count: int = 0
    pca_mean: np.ndarray | None = None
    pca_std: np.ndarray | None = None
    pca_components: np.ndarray | None = None
    pca_explained_variance_ratio: np.ndarray | None = None

    def fit(self, train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> "TokenFeatureState":
        all_features = pd.concat(
            [train_frame[self.numeric_cols + self.categorical_cols], test_frame[self.numeric_cols + self.categorical_cols]],
            axis=0,
            ignore_index=True,
        )
        self.all_row_count = len(all_features)

        values = _safe_numeric_matrix(all_features, self.numeric_cols)
        if values.shape[1]:
            self.center = np.nanmedian(values, axis=0).astype("float32")
            self.low = np.nanquantile(values, 0.01, axis=0).astype("float32")
            self.high = np.nanquantile(values, 0.99, axis=0).astype("float32")
            self.high = np.where(self.high <= self.low, self.low + 1.0, self.high).astype("float32")
        else:
            self.center = np.zeros(0, dtype="float32")
            self.low = np.zeros(0, dtype="float32")
            self.high = np.ones(0, dtype="float32")

        self.qbin_edges = {}
        self.qbin_freq = {}
        self.rounded_freq_maps = {}
        for col in self.numeric_cols:
            col_values = pd.to_numeric(all_features[col], errors="coerce")
            valid = col_values.dropna().to_numpy(dtype="float32")
            if len(valid) == 0:
                edges = np.asarray([-np.inf, np.inf], dtype="float32")
            else:
                raw_edges = np.nanquantile(valid, np.linspace(0, 1, self.n_bins + 1))
                edges = np.unique(raw_edges.astype("float32"))
                if len(edges) < 2:
                    edges = np.asarray([float(valid.min()) - 1.0, float(valid.max()) + 1.0], dtype="float32")
                edges[0] = -np.inf
                edges[-1] = np.inf
            self.qbin_edges[col] = edges

            ids = self.qbin_ids(all_features, col)
            n_bins = max(1, len(edges) - 1)
            valid_ids = ids[ids >= 0]
            counts = np.bincount(valid_ids, minlength=n_bins).astype("float32")
            self.qbin_freq[col] = counts / max(1.0, float(len(ids)))

            for decimals in (2, 3):
                rounded = col_values.round(decimals)
                freq_map = rounded.value_counts(dropna=True, normalize=True).to_dict()
                self.rounded_freq_maps[(col, decimals)] = {float(k): float(v) for k, v in freq_map.items()}

        self.category_values = {}
        self.category_maps = {}
        self.category_frequency_maps = {}
        for col in self.categorical_cols:
            values = _normalized_category_series(all_features, col)
            counts = values.value_counts(dropna=False)
            categories = counts.index.astype(str).tolist()
            self.category_values[col] = categories
            self.category_maps[col] = {category: idx for idx, category in enumerate(categories)}
            self.category_frequency_maps[col] = {str(k): float(v) / max(1.0, float(len(values))) for k, v in counts.items()}

        self._fit_pca(train_frame, test_frame)
        return self

    def transform_scaled(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        x = _safe_numeric_matrix(frame, self.numeric_cols)
        if x.shape[1] == 0:
            return x, x
        missing = np.isnan(x).astype("float32")
        x = np.where(np.isnan(x), self.center.reshape(1, -1), x)
        x = np.clip(x, self.low.reshape(1, -1), self.high.reshape(1, -1))
        scaled = 2.0 * (x - self.low.reshape(1, -1)) / (self.high - self.low).reshape(1, -1) - 1.0
        return scaled.astype("float32"), missing

    def qbin_ids(self, frame: pd.DataFrame, col: str) -> np.ndarray:
        values = pd.to_numeric(frame[col], errors="coerce").to_numpy(dtype="float32")
        edges = self.qbin_edges[col]
        ids = np.searchsorted(edges[1:-1], values, side="right").astype("int16")
        ids[np.isnan(values)] = -1
        return ids

    def category_codes_and_freqs(self, frame: pd.DataFrame, col: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        values = _normalized_category_series(frame, col)
        mapping = self.category_maps[col]
        freq_map = self.category_frequency_maps[col]
        codes = values.map(mapping).fillna(-1).to_numpy(dtype="int16")
        freqs = values.map(freq_map).fillna(0.0).to_numpy(dtype="float32")
        missing = (values == MISSING_TOKEN).to_numpy(dtype="float32")
        return codes, freqs, missing

    def _fit_pca(self, train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> None:
        rng = np.random.default_rng(self.seed)
        n_train = min(len(train_frame), self.pca_sample_n // 2)
        n_test = min(len(test_frame), self.pca_sample_n - n_train)
        train_idx = rng.choice(len(train_frame), size=n_train, replace=False)
        test_idx = rng.choice(len(test_frame), size=n_test, replace=False)
        sample = pd.concat(
            [train_frame.iloc[train_idx], test_frame.iloc[test_idx]],
            axis=0,
            ignore_index=True,
        )

        source, _ = build_flat_target_free_features(sample, self, include_pca=False)
        if source.shape[1] == 0:
            self.pca_mean = np.zeros((1, 0), dtype="float32")
            self.pca_std = np.ones((1, 0), dtype="float32")
            self.pca_components = np.zeros((0, 0), dtype="float32")
            self.pca_explained_variance_ratio = np.zeros(0, dtype="float32")
            return

        self.pca_mean = source.mean(axis=0, keepdims=True).astype("float32")
        self.pca_std = source.std(axis=0, keepdims=True).astype("float32")
        self.pca_std = np.where(self.pca_std <= 1e-6, 1.0, self.pca_std).astype("float32")
        x = ((source - self.pca_mean) / self.pca_std).astype("float32")

        cov = (x.T @ x) / max(1, x.shape[0] - 1)
        eigvals, eigvecs = np.linalg.eigh(cov.astype("float64"))
        order = np.argsort(eigvals)[::-1]
        n_components = min(self.pca_components_count, len(order))
        order = order[:n_components]
        self.pca_components = eigvecs[:, order].T.astype("float32")
        total = max(float(np.maximum(eigvals, 0.0).sum()), 1e-12)
        self.pca_explained_variance_ratio = (np.maximum(eigvals[order], 0.0) / total).astype("float32")

    def transform_pca(self, base_features: np.ndarray) -> tuple[np.ndarray, list[str]]:
        if self.pca_components is None or self.pca_components.shape[0] == 0:
            return np.zeros((len(base_features), 0), dtype="float32"), []
        x = ((base_features - self.pca_mean) / self.pca_std).astype("float32")
        z = x @ self.pca_components.T
        reconstruction = z @ self.pca_components
        reconstruction_error = np.mean((x - reconstruction) ** 2, axis=1, keepdims=True).astype("float32")
        names = [f"pca_svd__component_{i}" for i in range(z.shape[1])] + ["pca_svd__reconstruction_error"]
        return np.concatenate([z.astype("float32"), reconstruction_error], axis=1), names


## 5. Token Builders

There are two output views:

- `build_numeric_token_tensor` and `build_categorical_token_tensor`: field-token tensors for the transformer.
- `build_flat_target_free_features`: a wide 2D token matrix for XGBoost.


### セル7: 数値トークンテンソルと平坦特徴行列の構築

**何をしているか**
`build_numeric_token_tensor` は、形が **(行数, 数値フィールド数, トークン次元)** の3次元テンソルを作ります。各数値フィールド1つにつき1トークン、その中身が以下の成分:

| 成分 | 内容 |
|---|---|
| `scaled` | ロバスト標準化した値 |
| `missing` | 欠損フラグ |
| `scaled_square`, `abs_scaled` | 2乗、絶対値 |
| `sin_f1..f8`, `cos_f1..f8` | `sin(π · scaled · f)`, `cos(π · scaled · f)`(f = 1,2,4,8)の**フーリエ基底** |
| 局所基底 | 中心 −1.0, −0.5, 0, 0.5, 1.0 に置いたガウス的な**局所反応関数** |
| 頻度系 | 分位ビンの頻度、丸め値の頻度 |

欠損行では `active = 1 - missing` を掛けて基底成分を0にします。
`build_flat_target_free_features` は同じ情報を**横に全部つないだ2次元行列**にして XGBoost 用に出します。

**なぜそうするのか**
- **なぜフーリエ基底を足すのか**: ニューラルネットは**滑らかな低周波関数へのバイアス(spectral bias)**が強く、生の数値1個からは「所得の細かい変化に応じた急な反応」を学びにくいことが知られています。`sin/cos` を周波数違いで与えると、**線形結合だけで細かい凹凸を表現できる**ようになります。NeRF の positional encoding や Random Fourier Features と同じ発想を、表形式データに持ち込んだものです。
- **なぜ局所ガウス基底も足すのか**: フーリエ基底は**大域的**(1点の変化が全域に効く)なので、「年収が中央値付近のときだけ挙動が違う」といった**局所的な効果**の表現は苦手です。特定の位置に山を持つ局所基底を併用すると、大域と局所の両方をカバーできます。RBFネットワークやスプライン基底と同じ役割です。
- **なぜ欠損フラグを掛けて基底を0にするのか**: 欠損を0で埋めると、`sin(0)=0, cos(0)=1` となり「値が0だった人」と区別がつかなくなります。`active` を掛けて基底を丸ごと殺し、代わりに `missing=1` というトークン成分で「欠損である」ことを別次元で伝えます。**欠損は"値"ではなく"状態"として扱う**という正しい設計です。
- **なぜ XGBoost 用に平らな行列も作るのか**: 木モデルは1つの特徴量のしきい値で分割するので、**テンソルの構造(どのフィールドのどの成分か)を理解できません**。同じ情報を横に並べた形が必要です。**同じ特徴量から、モデルの帰納バイアスに合わせた2つのビューを作る**のがこの notebook の中核設計です。


In [ ]:
def build_numeric_token_tensor(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    scaled, missing = state.transform_scaled(frame)
    cols = state.numeric_cols
    if not cols:
        return np.zeros((len(frame), 0, 0), dtype="float32"), []

    active = 1.0 - missing
    token_pieces: list[np.ndarray] = []
    token_names: list[str] = []

    token_pieces.append(scaled[:, :, None])
    token_names.append("scaled")
    token_pieces.append(missing[:, :, None])
    token_names.append("missing")
    token_pieces.append((scaled**2)[:, :, None])
    token_names.append("scaled_square")
    token_pieces.append(np.abs(scaled)[:, :, None])
    token_names.append("abs_scaled")

    freqs = np.asarray(state.frequencies, dtype="float32")
    phase = np.pi * scaled[:, :, None] * freqs.reshape(1, 1, -1)
    token_pieces.append(np.sin(phase).astype("float32") * active[:, :, None])
    token_names.extend([f"sin_f{int(freq)}" for freq in state.frequencies])
    token_pieces.append(np.cos(phase).astype("float32") * active[:, :, None])
    token_names.extend([f"cos_f{int(freq)}" for freq in state.frequencies])

    centers = np.asarray(state.local_centers, dtype="float32")
    local_width = 0.55
    local = np.maximum(1.0 - np.abs(scaled[:, :, None] - centers.reshape(1, 1, -1)) / local_width, 0.0)
    token_pieces.append(local.astype("float32") * active[:, :, None])
    token_names.extend([f"local_{center:+.2f}" for center in state.local_centers])

    qbin_id_values = []
    qbin_freq_values = []
    round2_freq_values = []
    round3_freq_values = []
    for col in cols:
        ids = state.qbin_ids(frame, col)
        n_bins = max(1, len(state.qbin_edges[col]) - 1)
        qbin_scaled = np.where(ids >= 0, ids / max(1.0, float(n_bins - 1)), 0.0).astype("float32")
        qbin_id_values.append(qbin_scaled)

        qbin_freq = np.zeros(len(frame), dtype="float32")
        valid = ids >= 0
        qbin_freq[valid] = state.qbin_freq[col][ids[valid]]
        qbin_freq_values.append(qbin_freq)

        values = pd.to_numeric(frame[col], errors="coerce")
        round2_freq_values.append(values.round(2).map(state.rounded_freq_maps[(col, 2)]).fillna(0.0).to_numpy(dtype="float32"))
        round3_freq_values.append(values.round(3).map(state.rounded_freq_maps[(col, 3)]).fillna(0.0).to_numpy(dtype="float32"))

    token_pieces.append(np.stack(qbin_id_values, axis=1)[:, :, None])
    token_names.append("qbin_id")
    token_pieces.append(np.stack(qbin_freq_values, axis=1)[:, :, None])
    token_names.append("qbin_freq")
    token_pieces.append(np.stack(round2_freq_values, axis=1)[:, :, None])
    token_names.append("rounded_freq_d2")
    token_pieces.append(np.stack(round3_freq_values, axis=1)[:, :, None])
    token_names.append("rounded_freq_d3")

    return np.concatenate(token_pieces, axis=2).astype("float32"), token_names


def build_categorical_token_tensor(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    cols = state.categorical_cols
    token_names = [
        "code_scaled",
        "missing",
        "frequency",
        "log_frequency",
        "one_minus_frequency",
        "is_rare",
        "field_cardinality",
        "active",
    ]
    if not cols:
        return np.zeros((len(frame), 0, len(token_names)), dtype="float32"), token_names

    pieces = []
    denom = math.log1p(max(1, state.all_row_count))
    for col in cols:
        codes, freqs, missing = state.category_codes_and_freqs(frame, col)
        n_categories = max(1, len(state.category_values[col]))
        code_scaled = np.where(codes >= 0, 2.0 * codes / max(1.0, float(n_categories - 1)) - 1.0, 0.0).astype("float32")
        log_frequency = (np.log1p(freqs * state.all_row_count) / denom).astype("float32")
        field_cardinality = np.full(len(frame), min(1.0, n_categories / MAX_ONE_HOT_CARDINALITY), dtype="float32")
        token = np.stack(
            [
                code_scaled,
                missing,
                freqs,
                log_frequency,
                1.0 - freqs,
                (freqs < CAT_RARE_FREQUENCY).astype("float32"),
                field_cardinality,
                1.0 - missing,
            ],
            axis=1,
        ).astype("float32")
        pieces.append(token)

    return np.stack(pieces, axis=1).astype("float32"), token_names


def build_categorical_one_hot(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    pieces: list[np.ndarray] = []
    names: list[str] = []

    for col in state.categorical_cols:
        categories = state.category_values[col]
        if len(categories) > MAX_ONE_HOT_CARDINALITY:
            continue
        codes, _, _ = state.category_codes_and_freqs(frame, col)
        one_hot = (codes[:, None] == np.arange(len(categories), dtype="int16").reshape(1, -1)).astype("float32")
        safe_names = [str(category).replace(" ", "_").replace("/", "_") for category in categories]
        pieces.append(one_hot)
        names.extend([f"onehot__{col}__{category}" for category in safe_names])

    if not pieces:
        return np.zeros((len(frame), 0), dtype="float32"), []
    return np.concatenate(pieces, axis=1).astype("float32"), names


def build_interaction_features(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    scaled, _ = state.transform_scaled(frame)
    cols = state.numeric_cols
    pieces: list[np.ndarray] = []
    names: list[str] = []

    for left_idx, left in enumerate(cols):
        for right_idx in range(left_idx + 1, len(cols)):
            right = cols[right_idx]
            left_values = scaled[:, left_idx]
            right_values = scaled[:, right_idx]
            diff = left_values - right_values
            _append(pieces, names, diff, [f"interact_diff__{left}__minus__{right}"])
            _append(pieces, names, left_values * right_values, [f"interact_product__{left}__x__{right}"])
            _append(pieces, names, np.abs(diff), [f"interact_absdiff__{left}__{right}"])

    col_pos = {col: idx for idx, col in enumerate(cols)}

    def has_num(*required: str) -> bool:
        return all(col in col_pos for col in required)

    if has_num("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"):
        home = scaled[:, col_pos["Charging_Stations_Near_Home"]]
        work = scaled[:, col_pos["Charging_Stations_Near_Work"]]
        _append(pieces, names, home + work, ["ev__charging_access_sum"])
        _append(pieces, names, home - work, ["ev__home_minus_work_charging"])
        _append(pieces, names, home * work, ["ev__home_x_work_charging"])

    if has_num("Daily_Commute_km", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"):
        commute = scaled[:, col_pos["Daily_Commute_km"]]
        charging_sum = scaled[:, col_pos["Charging_Stations_Near_Home"]] + scaled[:, col_pos["Charging_Stations_Near_Work"]]
        _append(pieces, names, commute - charging_sum, ["ev__commute_minus_charging_access"])
        _append(pieces, names, commute * charging_sum, ["ev__commute_x_charging_access"])

    if has_num("Annual_Income_USD", "Number_of_Cars_Owned"):
        income = scaled[:, col_pos["Annual_Income_USD"]]
        cars = scaled[:, col_pos["Number_of_Cars_Owned"]]
        _append(pieces, names, income - cars, ["ev__income_minus_car_count"])
        _append(pieces, names, income * cars, ["ev__income_x_car_count"])

    if has_num("Annual_Income_USD", "Environmental_Concern_Level"):
        income = scaled[:, col_pos["Annual_Income_USD"]]
        concern = scaled[:, col_pos["Environmental_Concern_Level"]]
        _append(pieces, names, income * concern, ["ev__income_x_environmental_concern"])

    one_hot, one_hot_names = build_categorical_one_hot(frame, state)
    if one_hot.shape[1] and scaled.shape[1]:
        for idx, col in enumerate(cols):
            products = one_hot * scaled[:, idx : idx + 1]
            product_names = [f"num_cat__{col}__x__{name}" for name in one_hot_names]
            _append(pieces, names, products, product_names)

    if not pieces:
        return np.zeros((len(frame), 0), dtype="float32"), []
    return np.concatenate(pieces, axis=1).astype("float32"), names


def build_flat_target_free_features(
    frame: pd.DataFrame,
    state: TokenFeatureState,
    include_pca: bool = True,
) -> tuple[np.ndarray, list[str]]:
    pieces: list[np.ndarray] = []
    names: list[str] = []

    numeric_tokens, numeric_token_names = build_numeric_token_tensor(frame, state)
    if numeric_tokens.shape[1]:
        pieces.append(numeric_tokens.reshape(len(frame), -1))
        names.extend(
            [
                f"numtoken__{col}__{token_name}"
                for col in state.numeric_cols
                for token_name in numeric_token_names
            ]
        )

    categorical_tokens, categorical_token_names = build_categorical_token_tensor(frame, state)
    if categorical_tokens.shape[1]:
        pieces.append(categorical_tokens.reshape(len(frame), -1))
        names.extend(
            [
                f"cattoken__{col}__{token_name}"
                for col in state.categorical_cols
                for token_name in categorical_token_names
            ]
        )

    one_hot, one_hot_names = build_categorical_one_hot(frame, state)
    if one_hot.shape[1]:
        pieces.append(one_hot)
        names.extend(one_hot_names)

    interaction_block, interaction_names = build_interaction_features(frame, state)
    if interaction_block.shape[1]:
        pieces.append(interaction_block)
        names.extend(interaction_names)

    features = np.concatenate(pieces, axis=1).astype("float32") if pieces else np.zeros((len(frame), 0), dtype="float32")

    if include_pca:
        pca_features, pca_names = state.transform_pca(features)
        if pca_features.shape[1]:
            features = np.concatenate([features, pca_features], axis=1).astype("float32")
            names.extend(pca_names)

    return features, names


### セル8: カテゴリトークンテンソル(V2:カテゴリの同一性を強化)

**何をしているか**
カテゴリ列1つにつき1トークンを作ります。トークンの成分は12個:

`code_scaled`(カテゴリIDを −1〜+1 に線形写像)、`missing`、`frequency`、`log_frequency`、`one_minus_frequency`、`is_rare`、`field_cardinality`、`active`、そして `cat_identity_0..3` という**4スロットの one-hot 的な同一性ビット**。

**なぜそうするのか**
- **なぜ `code_scaled` だけでは足りないのか**: カテゴリIDを数値化すると「Urban=0, Suburban=1, Rural=2」のような**存在しない順序**をモデルに教えてしまいます。木モデルは分割で吸収できますが、Transformer の線形射影は「0.5 は Urban と Suburban の中間」と解釈してしまいます。だから `cat_identity_*` という**順序を持たない同一性表現**を併記しています。
- **なぜ one-hot が4スロットだけなのか**: 全カテゴリ分の one-hot を持つとトークン次元がカテゴリ数に比例して膨らみ、フィールドごとに次元が変わってしまいます(トークンは同じ次元でないと同じ射影行列に通せません)。**上位数個+ハッシュ的な割り当てで固定4次元に押し込む**という妥協です。低濃度のカテゴリが多いこのデータでは実用上十分です。
- **なぜ頻度を複数の形(生・log・1−freq)で持つのか**: 生の頻度は上位カテゴリに支配され、log は裾を強調し、`1 - frequency` は「レアさ」を線形に表します。**同じ情報の異なる非線形変換を並べて、モデルに好きなものを選ばせる**のは GBDT では特に有効な戦略です(木は単調変換に不変ですが、分割点の探索しやすさは変わります)。
- **なぜ `is_rare` フラグを別に立てるのか**: `CAT_RARE_FREQUENCY=0.005` 未満のカテゴリは統計が不安定です。明示的にフラグを立てることで、モデルが「このサンプルの推定は信頼できない」を学習できます。**不確実性を特徴量として渡す**という考え方です。


In [ ]:
# V2 token: stronger categorical identity + EV semantic interaction

CAT_TOKEN_ONEHOT_SLOTS = 4

def build_categorical_token_tensor(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    base_names = [
        "code_scaled", "missing", "frequency", "log_frequency",
        "one_minus_frequency", "is_rare", "field_cardinality", "active",
    ]
    token_names = base_names + [f"cat_identity_{i}" for i in range(CAT_TOKEN_ONEHOT_SLOTS)]

    if not state.categorical_cols:
        return np.zeros((len(frame), 0, len(token_names)), dtype="float32"), token_names

    pieces = []
    denom = math.log1p(max(1, state.all_row_count))

    for col in state.categorical_cols:
        codes, freqs, missing = state.category_codes_and_freqs(frame, col)
        n_categories = max(1, len(state.category_values[col]))

        code_scaled = np.where(
            codes >= 0,
            2.0 * codes / max(1.0, float(n_categories - 1)) - 1.0,
            0.0,
        ).astype("float32")

        log_frequency = (np.log1p(freqs * state.all_row_count) / denom).astype("float32")
        field_cardinality = np.full(len(frame), min(1.0, n_categories / MAX_ONE_HOT_CARDINALITY), dtype="float32")

        identity = np.zeros((len(frame), CAT_TOKEN_ONEHOT_SLOTS), dtype="float32")
        valid = (codes >= 0) & (codes < CAT_TOKEN_ONEHOT_SLOTS)
        identity[np.arange(len(frame))[valid], codes[valid]] = 1.0

        base = np.stack(
            [
                code_scaled,
                missing,
                freqs,
                log_frequency,
                1.0 - freqs,
                (freqs < CAT_RARE_FREQUENCY).astype("float32"),
                field_cardinality,
                1.0 - missing,
            ],
            axis=1,
        ).astype("float32")

        pieces.append(np.concatenate([base, identity], axis=1))

    return np.stack(pieces, axis=1).astype("float32"), token_names


if "_ORIGINAL_BUILD_INTERACTION_FEATURES" not in globals():
    _ORIGINAL_BUILD_INTERACTION_FEATURES = build_interaction_features


def build_ev_semantic_features(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    def num(col):
        return pd.to_numeric(frame[col], errors="coerce").fillna(0.0).to_numpy(dtype="float32")

    def is_value(col, value):
        return (_normalized_category_series(frame, col).str.lower() == value.lower()).astype("float32").to_numpy()

    age = num("Age")
    income = num("Annual_Income_USD")
    commute = num("Daily_Commute_km")
    cars = num("Number_of_Cars_Owned")
    home = num("Charging_Stations_Near_Home")
    work = num("Charging_Stations_Near_Work")
    concern = num("Environmental_Concern_Level")

    income_log = np.log1p(np.maximum(income, 0.0))
    charging_total = home + work
    subsidy_yes = is_value("Subsidy_Available", "Yes")
    home_charge_yes = is_value("Home_Charging_Possible", "Yes")
    urban = is_value("City_Type", "Urban")
    rural = is_value("City_Type", "Rural")
    suv = is_value("Current_Car_Type", "SUV")
    truck = is_value("Current_Car_Type", "Truck")

    anxiety = _normalized_category_series(frame, "Range_Anxiety_Level").map(
        {"Low": 0.0, "Medium": 1.0, "High": 2.0}
    ).fillna(0.0).to_numpy(dtype="float32")

    feature_pairs = [
        ("ev_sem__charging_total", charging_total),
        ("ev_sem__work_minus_home_charging", work - home),
        ("ev_sem__charging_per_commute", charging_total / (1.0 + commute)),
        ("ev_sem__commute_per_charging", commute / (1.0 + charging_total)),
        ("ev_sem__income_log", income_log),
        ("ev_sem__income_per_car", income / (1.0 + cars)),
        ("ev_sem__income_x_subsidy", income_log * subsidy_yes),
        ("ev_sem__income_x_concern", income_log * concern),
        ("ev_sem__concern_minus_anxiety", concern - anxiety),
        ("ev_sem__concern_x_subsidy", concern * subsidy_yes),
        ("ev_sem__home_charge_x_charging", home_charge_yes * charging_total),
        ("ev_sem__home_charge_x_commute", home_charge_yes * commute),
        ("ev_sem__anxiety_x_commute", anxiety * commute),
        ("ev_sem__anxiety_x_charging", anxiety * charging_total),
        ("ev_sem__urban_x_charging", urban * charging_total),
        ("ev_sem__rural_x_commute", rural * commute),
        ("ev_sem__suv_x_income", suv * income_log),
        ("ev_sem__truck_x_commute", truck * commute),
        ("ev_sem__age_x_concern", age * concern),
    ]

    features = np.column_stack([values for _, values in feature_pairs]).astype("float32")
    names = [name for name, _ in feature_pairs]
    return features, names


def build_interaction_features(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    base, base_names = _ORIGINAL_BUILD_INTERACTION_FEATURES(frame, state)
    sem, sem_names = build_ev_semantic_features(frame, state)
    return np.concatenate([base, sem], axis=1).astype("float32"), base_names + sem_names

## 6. Leakage-Safe Target-Mean Tokens

These supervised tokens are fitted inside each fold. Validation rows never see their own labels during encoding.


### セル9: TargetMeanEncoder ― リーク安全な target encoding

**何をしているか**
目的変数を使う唯一のエンコーダです。**fold の学習部分だけで `fit` し、検証部分と test に `transform` する**という契約で使われます。

- 数値列 → 分位ビンごとの目的変数平均
- 丸めた数値ごとの目的変数平均
- カテゴリごとの目的変数平均
- **カテゴリのペア**ごとの目的変数平均

すべて**平滑化(smoothing)**されます。式はおおよそ:

```
encoded = (sum_of_target + prior * smoothing) / (count + smoothing)
```

`TARGET_SMOOTHING = 500.0`、ペアは `PAIR_TARGET_SMOOTHING = 1500.0`。

**なぜそうするのか**
- **target encoding とは**: カテゴリを「そのカテゴリにおける目的変数の平均」で置き換える手法。カテゴリの意味を1つの数値に凝縮でき、木モデルと相性が良い。
- **なぜリークするのか**: あるサンプルのカテゴリ平均に**そのサンプル自身のラベルが入っている**と、そのサンプルの特徴量が自分の答えを含んでしまいます。極端な例として、あるカテゴリに1サンプルしかなければ、その特徴量はラベルそのものです。学習データでは完璧に効き、テストでは無力 ―― **CVは異常に高く、LBは低い**という典型的な失敗になります。
- **なぜ平滑化するのか**: 出現回数が少ないカテゴリの平均は分散が大きすぎます。全体平均 `prior` に向かって縮める(**shrinkage**)ことで、「データが少ないほど全体平均を信じる」という経験ベイズ的な振る舞いになります。`smoothing=500` は「500件分の事前観測に相当する重みを prior に置く」という意味で、**かなり強めの平滑化**です。合成データで行数が多い(数十万行)ことを踏まえた設定でしょう。
- **なぜペアの平滑化がさらに強い(1500)のか**: 2つのカテゴリの組み合わせは**組み合わせ爆発でセルあたりの件数が激減する**ので、より強く縮める必要があります。次元が上がるほど平滑化を強くする、という原則です。
- **使うとどうなるか**: 効くときは強力(特に高濃度カテゴリ)ですが、**平滑化が弱いと必ず過学習します**。導入したら OOF AUC と LB の差(CV-LBギャップ)を必ず監視してください。


In [ ]:
@dataclass
class TargetMeanEncoder:
    state: TokenFeatureState
    smoothing: float = TARGET_SMOOTHING
    pair_smoothing: float = PAIR_TARGET_SMOOTHING
    rounded_decimals: int = 2
    prior: float | None = None
    qbin_maps: dict[str, np.ndarray] | None = None
    rounded_maps: dict[str, dict[float, float]] | None = None
    category_maps: dict[str, dict[str, float]] | None = None
    category_pair_maps: dict[tuple[str, str], dict[tuple[str, str], float]] | None = None

    def fit(self, frame: pd.DataFrame, target: np.ndarray) -> "TargetMeanEncoder":
        self.prior = float(np.mean(target))
        self.qbin_maps = {}
        self.rounded_maps = {}
        self.category_maps = {}
        self.category_pair_maps = {}

        for col in self.state.numeric_cols:
            ids = self.state.qbin_ids(frame, col)
            n_bins = max(1, len(self.state.qbin_edges[col]) - 1)
            valid = ids >= 0
            sums = np.bincount(ids[valid], weights=target[valid], minlength=n_bins).astype("float64")
            counts = np.bincount(ids[valid], minlength=n_bins).astype("float64")
            encoded = (sums + self.prior * self.smoothing) / (counts + self.smoothing)
            self.qbin_maps[col] = encoded.astype("float32")

            rounded = pd.to_numeric(frame[col], errors="coerce").round(self.rounded_decimals)
            tmp = pd.DataFrame({"key": rounded, "target": target})
            stats = tmp.dropna().groupby("key")["target"].agg(["sum", "count"])
            smooth = (stats["sum"] + self.prior * self.smoothing) / (stats["count"] + self.smoothing)
            self.rounded_maps[col] = {float(k): float(v) for k, v in smooth.items()}

        for col in self.state.categorical_cols:
            values = _normalized_category_series(frame, col)
            tmp = pd.DataFrame({"key": values, "target": target})
            stats = tmp.groupby("key")["target"].agg(["sum", "count"])
            smooth = (stats["sum"] + self.prior * self.smoothing) / (stats["count"] + self.smoothing)
            self.category_maps[col] = {str(k): float(v) for k, v in smooth.items()}

        for left_idx, left in enumerate(self.state.categorical_cols):
            for right in self.state.categorical_cols[left_idx + 1 :]:
                left_values = _normalized_category_series(frame, left)
                right_values = _normalized_category_series(frame, right)
                tmp = pd.DataFrame({"left": left_values, "right": right_values, "target": target})
                stats = tmp.groupby(["left", "right"])["target"].agg(["sum", "count"])
                smooth = (stats["sum"] + self.prior * self.pair_smoothing) / (stats["count"] + self.pair_smoothing)
                self.category_pair_maps[(left, right)] = {(str(k[0]), str(k[1])): float(v) for k, v in smooth.items()}

        return self

    def transform(self, frame: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
        pieces: list[np.ndarray] = []
        names: list[str] = []

        for col in self.state.numeric_cols:
            ids = self.state.qbin_ids(frame, col)
            encoded = np.full(len(frame), self.prior, dtype="float32")
            valid = ids >= 0
            encoded[valid] = self.qbin_maps[col][ids[valid]]
            _append(pieces, names, encoded - self.prior, [f"target_mean_delta_qbin__{col}"])

            rounded = pd.to_numeric(frame[col], errors="coerce").round(self.rounded_decimals)
            round_encoded = rounded.map(self.rounded_maps[col]).fillna(self.prior).to_numpy(dtype="float32")
            _append(pieces, names, round_encoded - self.prior, [f"target_mean_delta_rounded_d{self.rounded_decimals}__{col}"])

        for col in self.state.categorical_cols:
            values = _normalized_category_series(frame, col)
            encoded = values.map(self.category_maps[col]).fillna(self.prior).to_numpy(dtype="float32")
            _append(pieces, names, encoded - self.prior, [f"target_mean_delta_category__{col}"])

        for left_idx, left in enumerate(self.state.categorical_cols):
            for right in self.state.categorical_cols[left_idx + 1 :]:
                pair_map = self.category_pair_maps[(left, right)]
                left_values = _normalized_category_series(frame, left)
                right_values = _normalized_category_series(frame, right)
                encoded = np.fromiter(
                    (pair_map.get((str(lv), str(rv)), self.prior) for lv, rv in zip(left_values, right_values)),
                    dtype="float32",
                    count=len(frame),
                )
                _append(pieces, names, encoded - self.prior, [f"target_mean_delta_catpair__{left}__{right}"])

        if not pieces:
            return np.zeros((len(frame), 0), dtype="float32"), []
        return np.concatenate(pieces, axis=1).astype("float32"), names


def build_flat_design_matrix(
    frame: pd.DataFrame,
    state: TokenFeatureState,
    target_encoder: TargetMeanEncoder | None = None,
) -> tuple[np.ndarray, list[str]]:
    target_free, names = build_flat_target_free_features(frame, state, include_pca=True)
    if target_encoder is None:
        return target_free, names
    target_features, target_names = target_encoder.transform(frame)
    return np.concatenate([target_free, target_features], axis=1).astype("float32"), names + target_names


def build_oof_target_mean_features(
    frame: pd.DataFrame,
    target: np.ndarray,
    state: TokenFeatureState,
    seed: int,
) -> tuple[np.ndarray, list[str], TargetMeanEncoder]:
    inner_folds = make_stratified_folds(target, N_SPLITS, seed)
    full_encoder = TargetMeanEncoder(state=state).fit(frame, target)
    preview, target_names = full_encoder.transform(frame.head(1))
    oof_features = np.zeros((len(frame), preview.shape[1]), dtype="float32")

    for inner_fold, (inner_tr_idx, inner_va_idx) in enumerate(inner_folds, start=1):
        inner_encoder = TargetMeanEncoder(state=state).fit(frame.iloc[inner_tr_idx], target[inner_tr_idx])
        oof_features[inner_va_idx] = inner_encoder.transform(frame.iloc[inner_va_idx])[0]

    return oof_features, target_names, full_encoder


### セル10: V5パッチ ― マルチスケール分位ビン target encoding

**何をしているか**
既存の `TargetMeanEncoder` を**継承して上書き**し(`_TARGET_MEAN_BASE` に元クラスを退避してから同名で再定義)、数値列を **8 / 16 / 32 / 64 ビン**の4通りの粒度で分位ビン化して、それぞれで target encoding を作ります。

ビン境界は `np.nanquantile` で作り、両端を `-inf` / `+inf` に置き換えてから `np.searchsorted` でIDを振ります。NaN は `-1`。

**なぜそうするのか**
- **なぜ複数の粒度が必要か**: 20ビンで切ると、「所得の非常に細かい帯だけEV購入率が跳ねる」効果は平均に埋もれます。逆に64ビンにすると、1ビンあたりの件数が減って推定がノイズだらけになります。**どの粒度が正解かは列によって違う**ので、全部作ってモデルに選ばせるのが素直です。
  - これは信号処理の**マルチスケール解析**や、画像のピラミッド表現と同じ発想です。GBDT は特徴量が多くても分割で自然に取捨選択できるので、この戦略と相性が良い。
- **なぜ両端を ±inf にするのか**: test に train の分位範囲外の値が来たとき、`searchsorted` でビン範囲外のインデックスになって落ちるのを防ぎます。**境界を無限大に開いておけば、外挿値は必ず端のビンに入ります**。
- **なぜ継承で上書きするのか**: Kaggle の notebook は「V4のセルを消さずにV5のセルを足す」という育て方をされがちです。継承して `super().fit()` を呼んでから追加処理をすれば、**元の機能を壊さずに増築できます**。ただし同名クラスの再定義はセルの実行順に依存するので、**上から順に1回だけ実行する**前提が崩れると壊れます(改善点として後述)。


In [ ]:
# V5 target-mean patch: multi-scale numeric qbin TE

_TARGET_MEAN_BASE = globals().get("_BASE_TARGET_MEAN_ENCODER", TargetMeanEncoder)

MULTISCALE_TE_BINS = (8, 16, 32, 64)

class TargetMeanEncoder(_TARGET_MEAN_BASE):
    def fit(self, frame: pd.DataFrame, target: np.ndarray):
        super().fit(frame, target)
        self.multi_qbin_edges = {}
        self.multi_qbin_maps = {}

        for col in self.state.numeric_cols:
            values = pd.to_numeric(frame[col], errors="coerce").to_numpy(dtype="float32")

            for n_bins in MULTISCALE_TE_BINS:
                valid_values = values[~np.isnan(values)]
                if len(valid_values) == 0:
                    continue

                edges = np.unique(np.nanquantile(valid_values, np.linspace(0, 1, n_bins + 1)).astype("float32"))
                if len(edges) < 2:
                    continue

                edges[0] = -np.inf
                edges[-1] = np.inf

                ids = np.searchsorted(edges[1:-1], values, side="right").astype("int16")
                ids[np.isnan(values)] = -1

                valid = ids >= 0
                n_real_bins = max(1, len(edges) - 1)
                sums = np.bincount(ids[valid], weights=target[valid], minlength=n_real_bins).astype("float64")
                counts = np.bincount(ids[valid], minlength=n_real_bins).astype("float64")

                smoothing = self.smoothing * math.sqrt(n_bins / 20.0)
                encoded = (sums + self.prior * smoothing) / (counts + smoothing)

                self.multi_qbin_edges[(col, n_bins)] = edges
                self.multi_qbin_maps[(col, n_bins)] = encoded.astype("float32")

        return self

    def transform(self, frame: pd.DataFrame):
        base, base_names = super().transform(frame)
        pieces = [base]
        names = list(base_names)

        for col in self.state.numeric_cols:
            values = pd.to_numeric(frame[col], errors="coerce").to_numpy(dtype="float32")

            for n_bins in MULTISCALE_TE_BINS:
                edges = self.multi_qbin_edges.get((col, n_bins))
                mapping = self.multi_qbin_maps.get((col, n_bins))
                if edges is None or mapping is None:
                    continue

                ids = np.searchsorted(edges[1:-1], values, side="right").astype("int16")
                ids[np.isnan(values)] = -1

                encoded = np.full(len(frame), self.prior, dtype="float32")
                valid = ids >= 0
                encoded[valid] = mapping[ids[valid]]

                _append(
                    pieces,
                    names,
                    encoded - self.prior,
                    [f"target_mean_delta_multiscale_qbin{n_bins}__{col}"],
                )

        return np.concatenate(pieces, axis=1).astype("float32"), names

## 7. Fit Target-Free State and Inspect Blocks


### セル11: target-free 状態の学習と、特徴ブロックの点検

**何をしているか**
`TokenFeatureState(...).fit(train, test)` を実行して統計量を確定させ、train の先頭32行で各表現をプレビューします。平坦行列の形、数値トークンテンソルの形、カテゴリトークンテンソルの形、トークン軸の名前、PCAの寄与率、そして**特徴名を `__` の前で切って作った「ブロック別の特徴数」**を印字します。

**なぜそうするのか**
- **なぜ先頭32行だけで確認するのか**: 全データで作ると数分待たされます。**形状と名前の確認は32行で十分**で、間違いがあればここで即座に気づけます。「小さく試して形を確認してから全体に適用する」のは、重い特徴量エンジニアリングでの基本作法です。
- **なぜブロック別の特徴数を数えるのか**: このパイプラインは最終的に数千次元を生みます。「フーリエ基底が2000次元、target encoding が50次元」のように**どのブロックが次元を占領しているか**を把握しておかないと、あとで「特徴量を減らしたい」となったときに手が出せません。また、**意図せず0次元になっているブロック**(実装ミスで空になっている)にも気づけます。
- **なぜ PCA の寄与率を見るのか**: 上位10成分で分散の何割を説明できるかを見れば、**列同士がどれだけ冗長か**が分かります。寄与率が急速に落ちるなら独立性が高く、なだらかなら相関構造が強い。後者なら交互作用特徴量が効きやすい、という見立てにつながります。


In [ ]:
state = TokenFeatureState(numeric_cols=numeric_cols, categorical_cols=categorical_cols).fit(train, test)
preview_flat, preview_names = build_flat_design_matrix(train.head(32), state, None)
preview_numeric_tokens, numeric_token_axis_names = build_numeric_token_tensor(train.head(32), state)
preview_categorical_tokens, categorical_token_axis_names = build_categorical_token_tensor(train.head(32), state)

block_summary = (
    pd.Series(preview_names)
    .str.split("__", n=1)
    .str[0]
    .value_counts()
    .rename_axis("feature_block")
    .reset_index(name="n_features")
)

category_summary = pd.DataFrame(
    {
        "column": list(state.category_values.keys()),
        "n_categories": [len(values) for values in state.category_values.values()],
    }
)

print("Flat preview matrix:", preview_flat.shape)
print("Numeric token tensor:", preview_numeric_tokens.shape)
print("Categorical token tensor:", preview_categorical_tokens.shape)
print("Numeric token axis:", numeric_token_axis_names)
print("Categorical token axis:", categorical_token_axis_names)
print("PCA/SVD explained variance:", np.round(state.pca_explained_variance_ratio, 5).tolist())
display(block_summary)
display(category_summary)


### セル12: トークン特徴ブロックの可視化

**何をしているか**
train 先頭256行で平坦行列を作り、(左)ブロック別の特徴数の横棒グラフ、(右)**40行 × 先頭80特徴のヒートマップ**(`coolwarm`、`vmin=-1, vmax=+1`)を描きます。

**なぜそうするのか**
- **なぜヒートマップを見るのか**: 数値だけでは気づけない異常が目で分かります。具体的には ―― **列全体が真っ白(定数)**なら情報ゼロの特徴、**列が真っ赤/真っ青に振り切れている**ならスケーリングが効いていない、**縦縞のパターンが規則的すぎる**なら基底の周期が入力レンジと合っていない。これらは numpy の describe を眺めるより圧倒的に速く見つかります。
- **なぜ `vmin=-1, vmax=1` に固定するのか**: 自動スケールだと外れ値1つで全体が同じ色になり、何も見えなくなります。**標準化後の特徴は概ね [−1, 1] に入るはず**という設計上の期待を色域に固定することで、「はみ出しているもの」が視覚的に浮かび上がります。
- **一般則**: 特徴量エンジニアリングは書いた分だけバグが増えます。**作った特徴を必ず1回は目で見る**という習慣が、静かに壊れた特徴を数百次元抱えたまま学習する事故を防ぎます。


In [ ]:
# Token Feature Block Map

if MPL_AVAILABLE:
    full_preview, full_names = build_flat_design_matrix(train.head(256), state, None)

    block_counts = (
        pd.Series(full_names)
        .str.split("__", n=1)
        .str[0]
        .value_counts()
        .sort_values()
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].barh(block_counts.index, block_counts.values, color="#54A24B")
    axes[0].set_title("Target-Free Feature Blocks")
    axes[0].set_xlabel("number of features")

    sample = full_preview[:40]
    show_n = min(80, sample.shape[1])
    axes[1].imshow(sample[:, :show_n], aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
    axes[1].set_title(f"Token Matrix Preview: 40 rows x first {show_n} features")
    axes[1].set_xlabel("feature index")
    axes[1].set_ylabel("row index")

    plt.tight_layout()
    plt.show()

    block_counts_df = block_counts.rename("n_features").reset_index()
    block_counts_df.columns = ["block", "n_features"]
    display(block_counts_df)

## 7b. Forensics + Fold-Safe Combo TE

### セル13: V6パッチ ― forensic 特徴量と fold 安全なコンボ target encoding

**何をしているか**
2種類の追加を行います。

**(1) Forensic(法医学的)特徴量** ―― 数値の「作られ方」の痕跡を拾う特徴:
- `FORENSIC_ROUND_DECIMALS = (0,1,2,3)`: 各小数位で丸めた値そのもの、および丸め誤差。
- `FORENSIC_FREQ_DECIMALS = (2,3)`: 丸めた値の出現頻度。
- `FORENSIC_MODULI = (2,5,10,100,1000)`: 値を各法で割った剰余(`value % 100` など)。

**(2) コンボ target encoding** ―― ドメイン知識で選んだ意味のあるペアに限定して target encoding:
- 数値×カテゴリ: `(環境意識, 補助金の有無)`、`(年収, 現在の車種)`、`(通勤距離, 航続距離不安)`、`(自宅充電可否, 職場付近の充電設備)` など13組。
- 数値×数値: `(年収, 環境意識)`、`(通勤距離, 充電ステーション数)` など。

**なぜそうするのか**
- **forensic 特徴量とは何で、なぜ効くのか**: Playground のデータは**実データから合成器で生成**されています。生成器は内部で乱数を丸めたり、離散分布からサンプリングしたりするので、**「値がちょうど100の倍数である」「小数第2位が0である」といった痕跡**が残ることがあります。そしてその痕跡が生成時のクラス条件と相関していると、**モデルにとっては強力な手がかり**になります。
  - **いつ使うか**: 合成データ(Playgroundシリーズ)、あるいは「複数のソースを結合したデータ」で有効。**実データ一本のコンペではほぼ無意味かノイズ**です。
  - **使うとどうなるか**: 当たれば数千分の1〜数百分の1のAUC向上。外れれば数百次元のノイズが増え、XGBoost の学習が遅くなり過学習リスクが上がります。**必ず OOF で入れる前後を比較してください**。倫理的にも「データの穴を突いている」だけで汎化する知識ではない、という自覚は必要です。
- **なぜペアを全組み合わせにせず手で選ぶのか**: 数値10列 × カテゴリ7列で70組、さらに数値同士で45組。全部作ると**次元が爆発し、平滑化しても過学習します**。「環境意識が高くて補助金がある人」「通勤距離が長くて自宅充電できない人」のように、**EV購入の意思決定として意味が通るペアだけ**に絞っています。ドメイン知識を特徴選択に使う正攻法です。
- **なぜ「fold安全」を強調するのか**: これらはすべて目的変数を使う特徴量なので、**fold の学習部分だけで統計を取る**必要があります。セル9の契約をそのまま踏襲しています。


In [ ]:
# V6 PDF-inspired forensic features + stronger leakage-safe TE

FORENSIC_ROUND_DECIMALS = (0, 1, 2, 3)
FORENSIC_FREQ_DECIMALS = (2, 3)
FORENSIC_MODULI = (2, 5, 10, 100, 1000)
FORENSIC_EPS = 1e-6

NUM_CAT_TE_SPECS = [
    ("Environmental_Concern_Level", "Subsidy_Available"),
    ("Environmental_Concern_Level", "Range_Anxiety_Level"),
    ("Environmental_Concern_Level", "Home_Charging_Possible"),
    ("Annual_Income_USD", "Subsidy_Available"),
    ("Annual_Income_USD", "Current_Car_Type"),
    ("Annual_Income_USD", "City_Type"),
    ("Daily_Commute_km", "Range_Anxiety_Level"),
    ("Daily_Commute_km", "Home_Charging_Possible"),
    ("Daily_Commute_km", "City_Type"),
    ("Charging_Stations_Near_Home", "Home_Charging_Possible"),
    ("Charging_Stations_Near_Work", "City_Type"),
    ("Number_of_Cars_Owned", "Current_Car_Type"),
    ("Age", "City_Type"),
]

NUM_NUM_TE_SPECS = [
    ("Annual_Income_USD", "Environmental_Concern_Level"),
    ("Daily_Commute_km", "Environmental_Concern_Level"),
    ("Daily_Commute_km", "Charging_Stations_Near_Home"),
    ("Daily_Commute_km", "Charging_Stations_Near_Work"),
]

CAT_TRIPLE_TE_SPECS = [
    ("City_Type", "Home_Charging_Possible", "Subsidy_Available"),
    ("Current_Car_Type", "Subsidy_Available", "Range_Anxiety_Level"),
    ("City_Type", "Current_Car_Type", "Range_Anxiety_Level"),
]


def _available_num_cat_specs(state):
    numeric = set(state.numeric_cols)
    categorical = set(state.categorical_cols)
    return [(num_col, cat_col) for num_col, cat_col in NUM_CAT_TE_SPECS if num_col in numeric and cat_col in categorical]


def _available_num_num_specs(state):
    numeric = set(state.numeric_cols)
    return [(left, right) for left, right in NUM_NUM_TE_SPECS if left in numeric and right in numeric]


def _available_cat_triple_specs(state):
    categorical = set(state.categorical_cols)
    return [cols for cols in CAT_TRIPLE_TE_SPECS if all(col in categorical for col in cols)]


def _string_key(parts: list[pd.Series]) -> pd.Series:
    key = parts[0].astype(str)
    for part in parts[1:]:
        key = key.str.cat(part.astype(str), sep="|")
    return key


def _qbin_key(frame: pd.DataFrame, state: TokenFeatureState, col: str) -> pd.Series:
    return pd.Series(state.qbin_ids(frame, col), index=frame.index).astype(str)


def _cat_key(frame: pd.DataFrame, col: str) -> pd.Series:
    return _normalized_category_series(frame, col).reset_index(drop=True)


def _fit_smooth_map(keys: pd.Series, target: np.ndarray, prior: float, smoothing: float) -> dict[str, float]:
    tmp = pd.DataFrame({"key": keys.astype(str).to_numpy(), "target": np.asarray(target, dtype="float32")})
    stats = tmp.groupby("key", observed=True)["target"].agg(["sum", "count"])
    encoded = (stats["sum"] + prior * smoothing) / (stats["count"] + smoothing)
    return {str(k): float(v) for k, v in encoded.items()}


def _apply_smooth_map(keys: pd.Series, mapping: dict[str, float], prior: float) -> np.ndarray:
    return keys.astype(str).map(mapping).fillna(prior).to_numpy(dtype="float32")


def _nearest_finite_edge_distance(values: np.ndarray, edges: np.ndarray, scale: float) -> np.ndarray:
    finite_edges = edges[np.isfinite(edges)]
    if len(finite_edges) == 0:
        return np.zeros(len(values), dtype="float32")
    pos = np.searchsorted(finite_edges, values, side="left")
    left_idx = np.clip(pos - 1, 0, len(finite_edges) - 1)
    right_idx = np.clip(pos, 0, len(finite_edges) - 1)
    dist = np.minimum(np.abs(values - finite_edges[left_idx]), np.abs(values - finite_edges[right_idx]))
    return np.clip(dist / max(FORENSIC_EPS, scale), 0.0, 1.0).astype("float32")


def build_numeric_forensic_features(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    pieces: list[np.ndarray] = []
    names: list[str] = []
    if not state.numeric_cols:
        return np.zeros((len(frame), 0), dtype="float32"), []

    center = np.asarray(state.center, dtype="float32")
    low = np.asarray(state.low, dtype="float32")
    high = np.asarray(state.high, dtype="float32")

    for idx, col in enumerate(state.numeric_cols):
        raw = pd.to_numeric(frame[col], errors="coerce")
        values = raw.to_numpy(dtype="float32")
        valid = ~np.isnan(values)
        filled = np.where(valid, values, center[idx]).astype("float32")
        abs_filled = np.abs(filled).astype("float32")
        feature_scale = float(max(FORENSIC_EPS, high[idx] - low[idx]))

        for decimals in FORENSIC_ROUND_DECIMALS:
            rounded = np.round(filled, decimals)
            dist = np.abs(filled - rounded)
            dist_scaled = np.clip(dist * (2.0 * (10 ** decimals)), 0.0, 1.0).astype("float32")
            _append(pieces, names, dist_scaled, [f"forensic_round_dist_d{decimals}__{col}"])
            _append(pieces, names, (dist_scaled <= 1e-5).astype("float32"), [f"forensic_is_round_d{decimals}__{col}"])

        for decimals in FORENSIC_FREQ_DECIMALS:
            freq_map = state.rounded_freq_maps.get((col, decimals), {}) if state.rounded_freq_maps is not None else {}
            freq = raw.round(decimals).map(freq_map).fillna(0.0).to_numpy(dtype="float32")
            log_freq = (np.log1p(freq * max(1, state.all_row_count)) / math.log1p(max(1, state.all_row_count))).astype("float32")
            _append(pieces, names, freq, [f"forensic_rounded_freq_d{decimals}__{col}"])
            _append(pieces, names, log_freq, [f"forensic_rounded_logfreq_d{decimals}__{col}"])

        qbin_ids = state.qbin_ids(frame, col)
        qbin_freq = np.zeros(len(frame), dtype="float32")
        valid_bins = qbin_ids >= 0
        qbin_freq[valid_bins] = state.qbin_freq[col][qbin_ids[valid_bins]]
        _append(pieces, names, qbin_freq, [f"forensic_qbin_freq__{col}"])
        _append(pieces, names, (qbin_freq < CAT_RARE_FREQUENCY).astype("float32"), [f"forensic_qbin_is_sparse__{col}"])
        edge_dist = _nearest_finite_edge_distance(filled, state.qbin_edges[col], feature_scale)
        _append(pieces, names, edge_dist, [f"forensic_nearest_qedge_dist__{col}"])

        frac = np.mod(abs_filled, 1.0).astype("float32")
        _append(pieces, names, frac, [f"forensic_fractional_part__{col}"])
        for modulus in FORENSIC_MODULI:
            mod_value = (np.mod(abs_filled, float(modulus)) / float(modulus)).astype("float32")
            close_to_mod = (np.minimum(mod_value, 1.0 - mod_value) <= 1e-5).astype("float32")
            _append(pieces, names, mod_value, [f"forensic_mod{modulus}_scaled__{col}"])
            _append(pieces, names, close_to_mod, [f"forensic_mod{modulus}_is_zero__{col}"])

        _append(pieces, names, (~valid).astype("float32"), [f"forensic_missing__{col}"])

    return np.concatenate(pieces, axis=1).astype("float32"), names


if "_V6_BASE_BUILD_FLAT_TARGET_FREE_FEATURES" not in globals():
    _V6_BASE_BUILD_FLAT_TARGET_FREE_FEATURES = build_flat_target_free_features

if "_V6_BASE_STATE_FIT_PCA" not in globals():
    _V6_BASE_STATE_FIT_PCA = TokenFeatureState._fit_pca


def _v6_fit_pca_with_base_features(self, train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> None:
    global build_flat_target_free_features
    current_builder = build_flat_target_free_features
    try:
        build_flat_target_free_features = _V6_BASE_BUILD_FLAT_TARGET_FREE_FEATURES
        return _V6_BASE_STATE_FIT_PCA(self, train_frame, test_frame)
    finally:
        build_flat_target_free_features = current_builder


TokenFeatureState._fit_pca = _v6_fit_pca_with_base_features


def build_flat_target_free_features(
    frame: pd.DataFrame,
    state: TokenFeatureState,
    include_pca: bool = True,
) -> tuple[np.ndarray, list[str]]:
    base_features, base_names = _V6_BASE_BUILD_FLAT_TARGET_FREE_FEATURES(frame, state, include_pca=include_pca)
    if not include_pca:
        return base_features, base_names
    forensic_features, forensic_names = build_numeric_forensic_features(frame, state)
    if forensic_features.shape[1] == 0:
        return base_features, base_names
    return np.concatenate([base_features, forensic_features], axis=1).astype("float32"), base_names + forensic_names


if "_V6_BASE_TARGET_MEAN_ENCODER" not in globals():
    _V6_BASE_TARGET_MEAN_ENCODER = TargetMeanEncoder


class TargetMeanEncoder(_V6_BASE_TARGET_MEAN_ENCODER):
    def fit(self, frame: pd.DataFrame, target: np.ndarray):
        super().fit(frame, target)
        self.num_cat_maps = {}
        self.num_num_maps = {}
        self.cat_triple_maps = {}

        for num_col, cat_col in _available_num_cat_specs(self.state):
            keys = _string_key([_qbin_key(frame, self.state, num_col).reset_index(drop=True), _cat_key(frame, cat_col)])
            self.num_cat_maps[(num_col, cat_col)] = _fit_smooth_map(keys, target, self.prior, self.pair_smoothing)

        for left, right in _available_num_num_specs(self.state):
            keys = _string_key([
                _qbin_key(frame, self.state, left).reset_index(drop=True),
                _qbin_key(frame, self.state, right).reset_index(drop=True),
            ])
            self.num_num_maps[(left, right)] = _fit_smooth_map(keys, target, self.prior, self.pair_smoothing * 1.25)

        for cols in _available_cat_triple_specs(self.state):
            keys = _string_key([_cat_key(frame, col) for col in cols])
            self.cat_triple_maps[cols] = _fit_smooth_map(keys, target, self.prior, self.pair_smoothing * 1.75)

        return self

    def transform(self, frame: pd.DataFrame):
        base, base_names = super().transform(frame)
        pieces = [base]
        names = list(base_names)

        for (num_col, cat_col), mapping in self.num_cat_maps.items():
            keys = _string_key([_qbin_key(frame, self.state, num_col).reset_index(drop=True), _cat_key(frame, cat_col)])
            encoded = _apply_smooth_map(keys, mapping, self.prior)
            _append(pieces, names, encoded - self.prior, [f"target_mean_delta_numcat__{num_col}__{cat_col}"])

        for (left, right), mapping in self.num_num_maps.items():
            keys = _string_key([
                _qbin_key(frame, self.state, left).reset_index(drop=True),
                _qbin_key(frame, self.state, right).reset_index(drop=True),
            ])
            encoded = _apply_smooth_map(keys, mapping, self.prior)
            _append(pieces, names, encoded - self.prior, [f"target_mean_delta_numpair__{left}__{right}"])

        for cols, mapping in self.cat_triple_maps.items():
            keys = _string_key([_cat_key(frame, col) for col in cols])
            encoded = _apply_smooth_map(keys, mapping, self.prior)
            _append(pieces, names, encoded - self.prior, ["target_mean_delta_cattriple__" + "__".join(cols)])

        return np.concatenate(pieces, axis=1).astype("float32"), names


if "state" in globals():
    forensic_preview, forensic_names = build_numeric_forensic_features(train.head(256), state)
    te_preview_encoder = TargetMeanEncoder(state=state).fit(train.head(min(5000, len(train))), y[: min(5000, len(train))])
    te_preview, te_names = te_preview_encoder.transform(train.head(32))
    flat_preview_v6, flat_names_v6 = build_flat_target_free_features(train.head(32), state, include_pca=True)
    print("V6 forensic feature count:", forensic_preview.shape[1])
    print("V6 target-mean feature count:", te_preview.shape[1])
    print("V6 flat target-free feature count:", flat_preview_v6.shape[1])
    print("V6 num-cat TE specs:", _available_num_cat_specs(state))
    print("V6 num-num TE specs:", _available_num_num_specs(state))
    print("V6 cat-triple TE specs:", _available_cat_triple_specs(state))


## 7c.Score-Lift Patch: EV Priors, Combo Frequency, and Higher-Order TE

### セル14: V7パッチ ― EV事前知識、コンボ頻度、高次 target encoding

**何をしているか**
3つの追加:

1. **`V7_EXACT_NUMERIC_KEYS`** ―― `Number_of_Cars_Owned`, `Charging_Stations_Near_Home`, `Charging_Stations_Near_Work`, `Environmental_Concern_Level` の4列を、**連続量ではなく「正確な離散値」として**扱う。
2. **`V7_COMBO_FREQUENCY_SPECS`** ―― 2つ組・**3つ組**のカテゴリ組み合わせの**出現頻度**(目的変数を使わないので fold 外で計算可)。
3. **高次の fold 安全 target encoding** ―― 3つ組までの組み合わせに対する target encoding。

**なぜそうするのか**
- **なぜこの4列を「離散」扱いするのか**: 保有台数や近隣の充電ステーション数は、実際には**小さな整数しか取らない**(0,1,2,3…)。これを連続量として標準化すると、「2.5台」のような存在しない中間値を前提にした基底が作られます。**取りうる値が数個しかない列は、カテゴリとして扱った方が情報が保存されます**。セル3の `exact_overlap` で「test の値がほぼ train に含まれる」ことを確認済みなのが根拠になっています。
- **なぜ「頻度」は fold 外で作ってよいのか**: 出現回数の集計に**目的変数は一切登場しません**。だから train+test 全体で計算してもリークしません。セル6の `TokenFeatureState` と同じ原理です。**「この特徴はリークするか?」の判定基準は"目的変数を参照しているか"の1点だけ**、というのを覚えてください。
- **3つ組まで広げるとどうなるか**: 表現力は上がりますが、**セルあたりの件数が激減**します。`City_Type × Subsidy_Available × Range_Anxiety_Level` で3×2×3=18セルなら十分ですが、濃度が高い列を混ぜると数千セルになり、大半が数件しかない状態になります。だから平滑化が必須で、`PAIR_TARGET_SMOOTHING=1500` のような強い縮小が効いてきます。
- **なぜ「頻度」と「target平均」の両方を作るのか**: 頻度は「その組み合わせがどれくらい典型的か」、target平均は「その組み合わせだとどれくらい買いやすいか」を表し、**別の情報**です。頻度が低いのに target平均が極端、という組み合わせは「過学習の候補」であり、モデルはその2つを見比べて割り引くことができます。


In [ ]:
# V7 score-lift patch: EV priors + combo frequency + higher-order fold-safe TE

V7_EXACT_NUMERIC_KEYS = {
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
}

V7_COMBO_FREQUENCY_SPECS = [
    ("Subsidy_Available", "Range_Anxiety_Level"),
    ("Subsidy_Available", "Home_Charging_Possible"),
    ("Range_Anxiety_Level", "Home_Charging_Possible"),
    ("Current_Car_Type", "Subsidy_Available"),
    ("City_Type", "Subsidy_Available"),
    ("Environmental_Concern_Level", "Subsidy_Available"),
    ("Environmental_Concern_Level", "Range_Anxiety_Level"),
    ("Environmental_Concern_Level", "Home_Charging_Possible"),
    ("Subsidy_Available", "Range_Anxiety_Level", "Home_Charging_Possible"),
    ("City_Type", "Subsidy_Available", "Range_Anxiety_Level"),
    ("City_Type", "Home_Charging_Possible", "Subsidy_Available"),
    ("Current_Car_Type", "Subsidy_Available", "Range_Anxiety_Level"),
    ("Environmental_Concern_Level", "Subsidy_Available", "Range_Anxiety_Level"),
    ("Environmental_Concern_Level", "Subsidy_Available", "Home_Charging_Possible"),
]

V7_COMBO_ONEHOT_SPECS = [
    ("Subsidy_Available", "Range_Anxiety_Level", "Home_Charging_Possible", "Environmental_Concern_Level"),
]

V7_NUM_CATCAT_TE_SPECS = [
    ("Environmental_Concern_Level", "Subsidy_Available", "Range_Anxiety_Level"),
    ("Environmental_Concern_Level", "Subsidy_Available", "Home_Charging_Possible"),
    ("Annual_Income_USD", "Subsidy_Available", "Range_Anxiety_Level"),
    ("Annual_Income_USD", "Subsidy_Available", "Home_Charging_Possible"),
    ("Annual_Income_USD", "City_Type", "Subsidy_Available"),
    ("Annual_Income_USD", "Current_Car_Type", "Subsidy_Available"),
    ("Daily_Commute_km", "Range_Anxiety_Level", "Home_Charging_Possible"),
    ("Daily_Commute_km", "City_Type", "Range_Anxiety_Level"),
    ("Daily_Commute_km", "City_Type", "Subsidy_Available"),
    ("Charging_Stations_Near_Home", "Home_Charging_Possible", "Range_Anxiety_Level"),
    ("Charging_Stations_Near_Work", "City_Type", "Range_Anxiety_Level"),
    ("Age", "City_Type", "Subsidy_Available"),
]

V7_CAT_QUAD_TE_SPECS = [
    ("Subsidy_Available", "Range_Anxiety_Level", "Home_Charging_Possible", "City_Type"),
    ("Subsidy_Available", "Range_Anxiety_Level", "Home_Charging_Possible", "Current_Car_Type"),
    ("Subsidy_Available", "Range_Anxiety_Level", "Home_Charging_Possible", "Gender"),
]

V7_NUMNUMCAT_TE_SPECS = [
    ("Annual_Income_USD", "Environmental_Concern_Level", "Subsidy_Available"),
    ("Annual_Income_USD", "Daily_Commute_km", "Range_Anxiety_Level"),
    ("Daily_Commute_km", "Charging_Stations_Near_Home", "Home_Charging_Possible"),
    ("Daily_Commute_km", "Charging_Stations_Near_Work", "City_Type"),
]


def _v7_available_columns(state: TokenFeatureState) -> set[str]:
    return set(state.numeric_cols) | set(state.categorical_cols)


def _v7_available_specs(state: TokenFeatureState, specs: list[tuple[str, ...]]) -> list[tuple[str, ...]]:
    available = _v7_available_columns(state)
    return [tuple(spec) for spec in specs if all(col in available for col in spec)]


def _v7_numeric_key(frame: pd.DataFrame, state: TokenFeatureState, col: str) -> pd.Series:
    values = pd.to_numeric(frame[col], errors="coerce")
    if col in V7_EXACT_NUMERIC_KEYS:
        return values.round(0).astype("Int64").astype(str).replace("<NA>", MISSING_TOKEN).reset_index(drop=True)
    return pd.Series(state.qbin_ids(frame, col), index=frame.index).astype(str).reset_index(drop=True)


def _v7_key(frame: pd.DataFrame, state: TokenFeatureState, spec: tuple[str, ...]) -> pd.Series:
    parts = []
    for col in spec:
        if col in state.categorical_cols:
            parts.append(_cat_key(frame, col).reset_index(drop=True))
        else:
            parts.append(_v7_numeric_key(frame, state, col))
    return _string_key(parts)


def _v7_fit_combo_frequency_maps(train_frame: pd.DataFrame, test_frame: pd.DataFrame, state: TokenFeatureState):
    all_frame = pd.concat([train_frame, test_frame], axis=0, ignore_index=True)
    maps = {}
    onehot_maps = {}

    for spec in _v7_available_specs(state, V7_COMBO_FREQUENCY_SPECS):
        keys = _v7_key(all_frame, state, spec)
        freq = keys.value_counts(normalize=True, dropna=False)
        count = keys.value_counts(dropna=False)
        maps[spec] = {
            "frequency": {str(k): float(v) for k, v in freq.items()},
            "count": {str(k): int(v) for k, v in count.items()},
        }

    for spec in _v7_available_specs(state, V7_COMBO_ONEHOT_SPECS):
        keys = _v7_key(all_frame, state, spec)
        categories = keys.value_counts(dropna=False).index.astype(str).tolist()
        onehot_maps[spec] = {category: idx for idx, category in enumerate(categories)}

    return maps, onehot_maps


V7_COMBO_FREQUENCY_MAPS, V7_COMBO_ONEHOT_MAPS = _v7_fit_combo_frequency_maps(train, test, state)


def build_v7_combo_frequency_features(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    pieces: list[np.ndarray] = []
    names: list[str] = []
    denom = math.log1p(max(1, state.all_row_count))

    for spec, maps in V7_COMBO_FREQUENCY_MAPS.items():
        key = _v7_key(frame, state, spec)
        spec_name = "__".join(spec)
        freq = key.map(maps["frequency"]).fillna(0.0).to_numpy(dtype="float32")
        count = key.map(maps["count"]).fillna(0.0).to_numpy(dtype="float32")
        log_count = (np.log1p(count) / denom).astype("float32")
        _append(pieces, names, freq, [f"v7_combo_freq__{spec_name}"])
        _append(pieces, names, log_count, [f"v7_combo_logcount__{spec_name}"])
        _append(pieces, names, (freq < CAT_RARE_FREQUENCY).astype("float32"), [f"v7_combo_is_rare__{spec_name}"])

    for spec, mapping in V7_COMBO_ONEHOT_MAPS.items():
        key = _v7_key(frame, state, spec).astype(str)
        codes = key.map(mapping).fillna(-1).to_numpy(dtype="int32")
        one_hot = (codes[:, None] == np.arange(len(mapping), dtype="int32").reshape(1, -1)).astype("float32")
        spec_name = "__".join(spec)
        safe_categories = [str(category).replace(" ", "_").replace("/", "_") for category in mapping]
        _append(pieces, names, one_hot, [f"v7_combo_onehot__{spec_name}__{category}" for category in safe_categories])

    if not pieces:
        return np.zeros((len(frame), 0), dtype="float32"), []
    return np.concatenate(pieces, axis=1).astype("float32"), names


def build_v7_ev_prior_features(frame: pd.DataFrame, state: TokenFeatureState) -> tuple[np.ndarray, list[str]]:
    scaled, _ = state.transform_scaled(frame)
    pos = {col: idx for idx, col in enumerate(state.numeric_cols)}

    def scaled_num(col: str) -> np.ndarray:
        if col not in pos:
            return np.zeros(len(frame), dtype="float32")
        return scaled[:, pos[col]].astype("float32")

    def raw_num(col: str) -> np.ndarray:
        return pd.to_numeric(frame[col], errors="coerce").fillna(0.0).to_numpy(dtype="float32")

    def is_value(col: str, value: str) -> np.ndarray:
        if col not in frame.columns:
            return np.zeros(len(frame), dtype="float32")
        return (_normalized_category_series(frame, col).str.lower() == value.lower()).astype("float32").to_numpy()

    subsidy_yes = is_value("Subsidy_Available", "Yes")
    subsidy_no = is_value("Subsidy_Available", "No")
    home_yes = is_value("Home_Charging_Possible", "Yes")
    urban = is_value("City_Type", "Urban")
    rural = is_value("City_Type", "Rural")
    suv = is_value("Current_Car_Type", "SUV")
    truck = is_value("Current_Car_Type", "Truck")

    anxiety_level = _normalized_category_series(frame, "Range_Anxiety_Level").map(
        {"Low": 0.0, "Medium": 1.0, "High": 2.0}
    ).fillna(0.0).to_numpy(dtype="float32")
    anxiety_low = (anxiety_level == 0).astype("float32")
    anxiety_medium = (anxiety_level == 1).astype("float32")
    anxiety_high = (anxiety_level == 2).astype("float32")

    concern = raw_num("Environmental_Concern_Level")
    concern01 = np.clip((concern - 1.0) / 4.0, 0.0, 1.0).astype("float32")
    concern_high = (concern >= 4.0).astype("float32")
    concern_top = (concern >= 5.0).astype("float32")

    income_s = scaled_num("Annual_Income_USD")
    commute_s = scaled_num("Daily_Commute_km")
    age_s = scaled_num("Age")
    home_station_s = scaled_num("Charging_Stations_Near_Home")
    work_station_s = scaled_num("Charging_Stations_Near_Work")
    charge_access_s = home_station_s + work_station_s

    prior_score = (
        2.6 * subsidy_yes
        + 1.8 * concern01
        + 0.7 * home_yes
        + 0.5 * income_s
        + 0.35 * charge_access_s
        - 1.4 * (anxiety_level / 2.0)
        - 0.25 * commute_s
    ).astype("float32")
    prior_probability_shape = sigmoid_np(prior_score - 1.6)

    feature_pairs = [
        ("v7_prior__score", prior_score),
        ("v7_prior__probability_shape", prior_probability_shape),
        ("v7_prior__no_subsidy_or_high_anxiety", np.maximum(subsidy_no, anxiety_high)),
        ("v7_prior__subsidy_low_anxiety_home", subsidy_yes * anxiety_low * home_yes),
        ("v7_prior__subsidy_low_anxiety_concern", subsidy_yes * anxiety_low * concern01),
        ("v7_prior__subsidy_high_concern_income", subsidy_yes * concern_high * income_s),
        ("v7_prior__subsidy_top_concern_income", subsidy_yes * concern_top * income_s),
        ("v7_prior__no_subsidy_high_concern", subsidy_no * concern_high),
        ("v7_prior__medium_anxiety_subsidy", anxiety_medium * subsidy_yes),
        ("v7_prior__high_anxiety_subsidy", anxiety_high * subsidy_yes),
        ("v7_prior__home_charging_access", home_yes * charge_access_s),
        ("v7_prior__urban_charging_access", urban * charge_access_s),
        ("v7_prior__rural_commute_penalty", rural * commute_s),
        ("v7_prior__suv_income_concern", suv * income_s * concern01),
        ("v7_prior__truck_commute_anxiety", truck * commute_s * (anxiety_level / 2.0)),
        ("v7_prior__age_concern", age_s * concern01),
        ("v7_prior__income_minus_commute", income_s - commute_s),
        ("v7_prior__charging_minus_commute", charge_access_s - commute_s),
    ]

    features = np.column_stack([values for _, values in feature_pairs]).astype("float32")
    names = [name for name, _ in feature_pairs]
    return features, names


if "_V7_BASE_BUILD_FLAT_TARGET_FREE_FEATURES" not in globals():
    _V7_BASE_BUILD_FLAT_TARGET_FREE_FEATURES = build_flat_target_free_features


def build_flat_target_free_features(
    frame: pd.DataFrame,
    state: TokenFeatureState,
    include_pca: bool = True,
) -> tuple[np.ndarray, list[str]]:
    base_features, base_names = _V7_BASE_BUILD_FLAT_TARGET_FREE_FEATURES(frame, state, include_pca=include_pca)
    if not include_pca:
        return base_features, base_names
    prior_features, prior_names = build_v7_ev_prior_features(frame, state)
    combo_features, combo_names = build_v7_combo_frequency_features(frame, state)
    extras = [prior_features, combo_features]
    extra_names = prior_names + combo_names
    extras = [arr for arr in extras if arr.shape[1] > 0]
    if not extras:
        return base_features, base_names
    return np.concatenate([base_features, *extras], axis=1).astype("float32"), base_names + extra_names


if "_V7_BASE_TARGET_MEAN_ENCODER" not in globals():
    _V7_BASE_TARGET_MEAN_ENCODER = TargetMeanEncoder


class TargetMeanEncoder(_V7_BASE_TARGET_MEAN_ENCODER):
    def fit(self, frame: pd.DataFrame, target: np.ndarray):
        super().fit(frame, target)
        self.v7_num_catcat_maps = {}
        self.v7_cat_quad_maps = {}
        self.v7_num_num_cat_maps = {}

        for num_col, cat_left, cat_right in _v7_available_specs(self.state, V7_NUM_CATCAT_TE_SPECS):
            keys = _string_key([
                _qbin_key(frame, self.state, num_col).reset_index(drop=True),
                _cat_key(frame, cat_left),
                _cat_key(frame, cat_right),
            ])
            self.v7_num_catcat_maps[(num_col, cat_left, cat_right)] = _fit_smooth_map(
                keys, target, self.prior, self.pair_smoothing * 2.0
            )

        for cols in _v7_available_specs(self.state, V7_CAT_QUAD_TE_SPECS):
            keys = _string_key([_cat_key(frame, col) for col in cols])
            self.v7_cat_quad_maps[cols] = _fit_smooth_map(keys, target, self.prior, self.pair_smoothing * 2.5)

        for num_left, num_right, cat_col in _v7_available_specs(self.state, V7_NUMNUMCAT_TE_SPECS):
            keys = _string_key([
                _qbin_key(frame, self.state, num_left).reset_index(drop=True),
                _qbin_key(frame, self.state, num_right).reset_index(drop=True),
                _cat_key(frame, cat_col),
            ])
            self.v7_num_num_cat_maps[(num_left, num_right, cat_col)] = _fit_smooth_map(
                keys, target, self.prior, self.pair_smoothing * 2.25
            )

        return self

    def transform(self, frame: pd.DataFrame):
        base, base_names = super().transform(frame)
        pieces = [base]
        names = list(base_names)

        for (num_col, cat_left, cat_right), mapping in self.v7_num_catcat_maps.items():
            keys = _string_key([
                _qbin_key(frame, self.state, num_col).reset_index(drop=True),
                _cat_key(frame, cat_left),
                _cat_key(frame, cat_right),
            ])
            encoded = _apply_smooth_map(keys, mapping, self.prior)
            _append(pieces, names, encoded - self.prior, [f"v7_te_num_catcat__{num_col}__{cat_left}__{cat_right}"])

        for cols, mapping in self.v7_cat_quad_maps.items():
            keys = _string_key([_cat_key(frame, col) for col in cols])
            encoded = _apply_smooth_map(keys, mapping, self.prior)
            _append(pieces, names, encoded - self.prior, [f"v7_te_cat_quad__{'__'.join(cols)}"])

        for (num_left, num_right, cat_col), mapping in self.v7_num_num_cat_maps.items():
            keys = _string_key([
                _qbin_key(frame, self.state, num_left).reset_index(drop=True),
                _qbin_key(frame, self.state, num_right).reset_index(drop=True),
                _cat_key(frame, cat_col),
            ])
            encoded = _apply_smooth_map(keys, mapping, self.prior)
            _append(pieces, names, encoded - self.prior, [f"v7_te_num_num_cat__{num_left}__{num_right}__{cat_col}"])

        return np.concatenate(pieces, axis=1).astype("float32"), names


_v7_flat_preview, _v7_flat_names = build_flat_design_matrix(train.head(512), state, None)
_v7_target_preview, _v7_target_names = TargetMeanEncoder(state=state).fit(train.head(50_000), y[:50_000]).transform(train.head(512))
print("V7 combo frequency specs:", list(V7_COMBO_FREQUENCY_MAPS.keys()))
print("V7 combo one-hot specs:", {spec: len(mapping) for spec, mapping in V7_COMBO_ONEHOT_MAPS.items()})
print("V7 flat target-free feature count:", len(_v7_flat_names))
print("V7 target-mean feature count:", len(_v7_target_names))


## 8. XGBoost on Flat Token Features

XGBoost gets the wide 2D representation. Each fold uses a fold-specific target encoder, then predicts both validation rows and the test set.


### セル15: XGBoost の交差検証(fold内で target encoding を再構築)

**何をしているか**
`run_xgboost_cv()` の流れ:

1. `make_stratified_folds` で層化5分割。
2. GPU があれば `device="cuda"`。
3. **各 fold ごとに `TargetMeanEncoder` を「学習部分だけ」で fit** し、学習・検証・test を transform。
4. `early_stopping_rounds=200` 付きで学習(古い XGBoost 用の `fit(early_stopping_rounds=...)` フォールバックも用意)。
5. 検証部分の予測を OOF 配列に、test 予測を `1/N_SPLITS` ずつ加算。
6. fold ごとに AUC / logloss / Brier / 使われた木の本数(`best_iteration`)を記録。
7. 最後に**OOF全体のAUC**を印字。使い終わった行列は `del` + `gc.collect()`。

**なぜそうするのか**
- **なぜ fold ごとにエンコーダを作り直すのか(核心)**: これが**リークを防ぐ唯一の正しい方法**です。1回だけ全 train で target encoding を作って使い回すと、検証部分のサンプルは自分のラベルが混ざったエンコーディングを見ることになります。OOF AUC は 0.99 のような非現実的な値になり、LB は 0.94 のまま。**この乖離を「CVが信用できない」と誤解して迷走するのが、初心者が最も長く沼にはまるポイントです**。
- **なぜ test 予測を fold 平均するのか**: 5つのモデルそれぞれが test を予測し、平均を取ります。これは**バギング(bagging)**そのもので、分散が減ります。「fold 全体で再学習した1つのモデル」より安定することが多く、Kaggleの標準的なやり方です。
- **なぜ `best_iteration` を記録するのか**: fold 間で最適本数が大きくばらつく(例:800 と 4000)なら、**fold の難易度が不均一**か、early stopping のノイズが大きいサインです。学習率をさらに下げるか、fold 数を増やすかの判断材料になります。
- **なぜ `del` + `gc.collect()` を挟むのか**: 数千次元 × 数十万行の float32 行列を fold ごとに作るので、放っておくとメモリが積み上がります。Kaggle の 16〜30GB 制限では、**明示的な解放が実行完了の可否を分けます**。
- **なぜ AUC・logloss・Brier を3つとも記録するのか**: 再掲になりますが、AUC は順位のみ、logloss と Brier は確率の質を見ます。**ブレンドの段階で logloss をタイブレークに使うので、ここで記録しておく必要があります**。


In [ ]:
def infer_xgb_device() -> str:
    if TORCH_AVAILABLE:
        try:
            import torch
            if torch.cuda.is_available():
                return "cuda"
        except Exception:
            pass
    return "cpu"


def prediction_report(y_true: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    pred = clip_prob(pred)
    return {
        "auc": roc_auc_score_np(y_true, pred),
        "logloss": binary_log_loss(y_true, pred),
        "brier": brier_score(y_true, pred),
    }


def run_xgboost_cv() -> dict[str, object] | None:
    if not RUN_XGBOOST:
        print("RUN_XGBOOST=False; skipping XGBoost.")
        return None
    if not XGBOOST_AVAILABLE:
        print("xgboost is not installed in this environment; skipping XGBoost.")
        return None

    from xgboost import XGBClassifier

    folds = make_stratified_folds(y, N_SPLITS, SEED)
    xgb_device = infer_xgb_device()
    print("XGBoost device:", xgb_device)

    base_params = dict(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=XGB_N_ESTIMATORS,
        learning_rate=XGB_LEARNING_RATE,
        max_depth=6,
        min_child_weight=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=8.0,
        tree_method="hist",
        device=xgb_device,
        random_state=SEED,
        n_jobs=-1,
        early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS,
    )

    oof = np.zeros(len(train), dtype="float32")
    test_pred = np.zeros(len(test), dtype="float32")
    fold_rows = []
    feature_names = None

    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        fold_start = time.time()
        print(f"\nXGBoost fold {fold}/{N_SPLITS}")
        fold_train = train.iloc[tr_idx].reset_index(drop=True)
        fold_target = y[tr_idx]
        target_tr, target_names, encoder = build_oof_target_mean_features(
            fold_train,
            fold_target,
            state,
            seed=SEED + fold,
        )

        x_tr_base, base_names = build_flat_target_free_features(fold_train, state, include_pca=True)
        target_va, _ = encoder.transform(train.iloc[va_idx])
        target_test, _ = encoder.transform(test)
        x_va_base, _ = build_flat_target_free_features(train.iloc[va_idx], state, include_pca=True)
        x_test_base, _ = build_flat_target_free_features(test, state, include_pca=True)

        x_tr = np.concatenate([x_tr_base, target_tr], axis=1).astype("float32")
        x_va = np.concatenate([x_va_base, target_va], axis=1).astype("float32")
        x_test = np.concatenate([x_test_base, target_test], axis=1).astype("float32")
        names = base_names + target_names

        if feature_names is None:
            feature_names = names
            print("XGBoost feature count:", len(feature_names))

        params = base_params.copy()
        if XGB_USE_SCALE_POS_WEIGHT:
            pos = max(1.0, float(fold_target.sum()))
            neg = max(1.0, float(len(fold_target) - fold_target.sum()))
            params["scale_pos_weight"] = neg / pos

        model = XGBClassifier(**params)
        try:
            model.fit(x_tr, y[tr_idx], eval_set=[(x_va, y[va_idx])], verbose=250)
        except TypeError:
            fallback_params = params.copy()
            fallback_params.pop("early_stopping_rounds", None)
            fallback_params.pop("device", None)
            model = XGBClassifier(**fallback_params)
            model.fit(
                x_tr,
                y[tr_idx],
                eval_set=[(x_va, y[va_idx])],
                early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS,
                verbose=250,
            )

        valid_pred = model.predict_proba(x_va)[:, 1].astype("float32")
        fold_test = model.predict_proba(x_test)[:, 1].astype("float32")
        oof[va_idx] = valid_pred
        test_pred += fold_test / N_SPLITS

        metrics = prediction_report(y[va_idx], valid_pred)
        best_iteration = getattr(model, "best_iteration", None)
        fold_rows.append(
            {
                "model": "xgboost_token_features",
                "fold": fold,
                **metrics,
                "best_iteration": int(best_iteration or 0),
            }
        )
        print(
            f"XGBoost fold {fold}: auc={metrics['auc']:.6f} "
            f"logloss={metrics['logloss']:.6f} brier={metrics['brier']:.6f} "
            f"runtime={time.time() - fold_start:.1f}s"
        )

        del x_tr, x_va, x_test, x_tr_base, x_va_base, x_test_base, target_tr, target_va, target_test, model
        gc.collect()

    fold_scores = pd.DataFrame(fold_rows)
    full_metrics = prediction_report(y, oof)
    print(f"\nXGBoost OOF AUC    : {full_metrics['auc']:.6f}")
    print(f"XGBoost OOF logloss: {full_metrics['logloss']:.6f}")
    print(f"XGBoost OOF Brier  : {full_metrics['brier']:.6f}")
    display(fold_scores)
    return {"name": "xgboost", "oof": oof, "test_pred": clip_prob(test_pred), "fold_scores": fold_scores, "feature_names": feature_names}


xgboost_result = run_xgboost_cv()


## 8b. XGBoost Variant Ensemble

## 9. TinyTokenTransformer Architecture

This model receives:

- one token per numeric field
- one token per categorical field
- one dense numeric/category interaction token
- one PCA/SVD geometry token
- one target-mean token

The `[CLS]` token summarizes the row after self-attention.


### セル16: TinyTokenTransformer のアーキテクチャ図を描く

**何をしているか**
実際に train 先頭16行を各ビルダーに通して**本物の形状を取得**し、それを箱と矢印の図として matplotlib で描画します。数値トークン数×成分数、カテゴリトークン数×成分数、交互作用ブロック、PCAブロック、target-meanブロックのサイズが図中に入ります。

**なぜそうするのか**
- **なぜ手描きの図ではなく、実データから形状を取るのか**: 図と実装がずれるのは notebook でよくある事故です。**実際に `.shape` を読んで図に書き込めば、図は常に実装と一致します**。「ドキュメントがコードから生成される」という考え方の、ごく小さな実践です。
- **なぜアーキテクチャを図示するのか**: このモデルは「数値フィールドトークン + カテゴリフィールドトークン + 密ブロック(交互作用/PCA/target-mean) + [CLS]」という**異種のトークンを1列に並べる**構造です。文章で書くと分かりにくいですが、図なら一目で分かります。読み手(将来の自分を含む)への投資です。
- **なぜ target-mean ブロックのプレビューに先頭5000行だけ使うのか**: 図を描くためだけに全データで fit するのは無駄です。形状さえ合っていればよいので、サンプルで十分。


In [ ]:
# TinyTokenTransformer Architecture Diagram

def plot_tiny_token_transformer_architecture():
    if not MPL_AVAILABLE:
        print("matplotlib is not available in this environment.")
        return

    preview_n = min(16, len(train))

    numeric_tokens, numeric_axis = build_numeric_token_tensor(train.head(preview_n), state)
    categorical_tokens, categorical_axis = build_categorical_token_tensor(train.head(preview_n), state)
    interaction_block, interaction_names = build_interaction_features(train.head(preview_n), state)

    pca_source, _ = build_flat_target_free_features(train.head(preview_n), state, include_pca=False)
    pca_block, pca_names = state.transform_pca(pca_source)

    preview_encoder = TargetMeanEncoder(state=state).fit(train.head(min(5000, len(train))), y[: min(5000, len(train))])
    target_mean_block, target_mean_names = preview_encoder.transform(train.head(preview_n))

    boxes = [
        ("Numeric field tokens", f"{numeric_tokens.shape[1]} fields x {numeric_tokens.shape[2]} features\nscaled, Fourier, local, frequency"),
        ("Categorical field tokens", f"{categorical_tokens.shape[1]} fields x {categorical_tokens.shape[2]} features\ncode, frequency, rarity"),
        ("Dense interaction token", f"{interaction_block.shape[1]} features\nnumeric pairs + num-category products"),
        ("PCA/SVD geometry token", f"{pca_block.shape[1]} features\nlow-rank row geometry"),
        ("Target-mean token", f"{target_mean_block.shape[1]} features\nOOF leakage-safe encodings"),
        ("Linear projections", f"each token -> d_model={TTT_D_MODEL}"),
        ("[CLS] + positional embedding", "row summary token + field positions"),
        ("Transformer encoder", f"{TTT_N_LAYERS} layers, {TTT_N_HEADS} heads\nself-attention across fields"),
        ("Classification head", "LayerNorm + Linear"),
        ("Output", "P(Will_Buy_EV = Yes)"),
    ]

    fig, ax = plt.subplots(figsize=(16, 8))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 8)
    ax.axis("off")

    positions = {
        0: (0.5, 5.9),
        1: (0.5, 4.55),
        2: (0.5, 3.2),
        3: (0.5, 1.85),
        4: (0.5, 0.5),
        5: (4.7, 3.25),
        6: (7.4, 3.25),
        7: (10.1, 3.25),
        8: (12.9, 3.25),
        9: (14.55, 3.25),
    }

    colors = {
        0: "#D7E8FA",
        1: "#FCE1C4",
        2: "#DDF0D6",
        3: "#E9D8F2",
        4: "#F8D7DA",
        5: "#E8E8E8",
        6: "#FFF2B8",
        7: "#D6EEF2",
        8: "#E8E8E8",
        9: "#D9EAD3",
    }

    def draw_box(idx, width=2.45, height=0.9):
        x, y0 = positions[idx]
        title, subtitle = boxes[idx]
        rect = plt.Rectangle(
            (x, y0),
            width,
            height,
            facecolor=colors[idx],
            edgecolor="#333333",
            linewidth=1.2,
            joinstyle="round",
        )
        ax.add_patch(rect)
        ax.text(x + width / 2, y0 + height * 0.64, title, ha="center", va="center", fontsize=10, fontweight="bold")
        ax.text(x + width / 2, y0 + height * 0.30, subtitle, ha="center", va="center", fontsize=8.5)

    for i in range(len(boxes)):
        draw_box(i, width=2.65 if i < 5 else 2.25)

    def arrow(start, end):
        sx, sy = start
        ex, ey = end
        ax.annotate(
            "",
            xy=(ex, ey),
            xytext=(sx, sy),
            arrowprops=dict(arrowstyle="->", color="#333333", lw=1.4, shrinkA=4, shrinkB=4),
        )

    for idx in range(5):
        x, y0 = positions[idx]
        arrow((x + 2.65, y0 + 0.45), (positions[5][0], positions[5][1] + 0.45))

    for idx in [5, 6, 7, 8]:
        x, y0 = positions[idx]
        next_x, next_y = positions[idx + 1]
        arrow((x + 2.25, y0 + 0.45), (next_x, next_y + 0.45))

    ax.text(
        8,
        7.45,
        "TinyTokenTransformer Architecture for S6E9 EV Purchase Prediction",
        ha="center",
        va="center",
        fontsize=16,
        fontweight="bold",
    )

    ax.text(
        8,
        6.95,
        "Tabular row -> field tokens + dense summary tokens -> self-attention -> positive-class probability",
        ha="center",
        va="center",
        fontsize=11,
        color="#444444",
    )

    plt.tight_layout()
    plt.show()


plot_tiny_token_transformer_architecture()

### セル17: TinyTokenTransformer 本体の定義

**何をしているか**
PyTorch がある場合のみ、以下の構造のモデルを定義します。

- **数値トークン**: `Linear(numeric_token_dim → d_model)` で全フィールド共通の射影。
- **カテゴリトークン**: `Linear(categorical_token_dim → d_model)` で同じく共通射影。
- **密ブロック**: 交互作用・PCA・target-mean の各ブロックをそれぞれ `Linear(dim → d_model)` で**1トークンずつに圧縮**。
- **[CLS] トークン**: 学習可能なベクトルを1つ先頭に置く。
- **Transformer エンコーダ**: `TTT_N_LAYERS` 層、`TTT_N_HEADS` ヘッド、dropout 付き。
- 最後に **[CLS] の出力**を分類ヘッドに通してロジット1つを出す。

**なぜそうするのか**
- **なぜ全フィールドで射影行列を共有するのか**: フィールドごとに別の `Linear` を持つとパラメータがフィールド数倍になり、**列が多いと即座に過学習します**。「トークンの中身の作り方(scaled / フーリエ / 頻度 …)は全フィールド共通の意味を持つ」ように設計されているので、共有射影が正当化されます。**特徴設計とモデル構造が対応している**わけです。ただし共有だけだとフィールドの区別がつかないので、実際には位置埋め込み的な情報がフィールド識別を担います。
- **なぜ [CLS] トークンを使うのか**: BERT と同じ手法です。全トークンの平均プーリングでもよいのですが、[CLS] は **attention を通じて「今回の予測にどのフィールドが重要か」を自分で重み付けして集約**できます。行ごとに重要なフィールドが違うタスク(高所得者は所得が効き、低所得者は充電設備が効く、など)では平均より有利です。
- **なぜ「Tiny」なのか**: 表形式データで大きな Transformer を使っても、GBDT に勝てないことがほとんどです。表データには「特徴間に空間的・逐次的構造がない」「サンプル数に比べて次元が低い」という性質があり、**帰納バイアスとして木の方が適している**からです。ここでの Transformer の役割は単独で勝つことではなく、**GBDT と誤りの傾向が異なるモデルを1本用意して、ブレンドで稼ぐ**ことです。だから小さくてよい。
- **dropout を入れる理由**: 表データの Transformer は容易に過学習します。dropout と早期打ち切りは必須の安全装置です。


In [ ]:
if TORCH_AVAILABLE:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    class TinyTokenTransformer(nn.Module):
        def __init__(
            self,
            n_numeric_tokens: int,
            numeric_token_dim: int,
            n_categorical_tokens: int,
            categorical_token_dim: int,
            dense_block_dims: list[int],
            d_model: int = TTT_D_MODEL,
            n_heads: int = TTT_N_HEADS,
            n_layers: int = TTT_N_LAYERS,
            dropout: float = TTT_DROPOUT,
        ):
            super().__init__()
            self.has_numeric = n_numeric_tokens > 0
            self.has_categorical = n_categorical_tokens > 0
            self.numeric_projection = nn.Linear(numeric_token_dim, d_model) if self.has_numeric else None
            self.categorical_projection = nn.Linear(categorical_token_dim, d_model) if self.has_categorical else None
            self.dense_projections = nn.ModuleList([nn.Linear(dim, d_model) for dim in dense_block_dims])

            n_tokens = 1 + n_numeric_tokens + n_categorical_tokens + len(dense_block_dims)
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
            self.position = nn.Parameter(torch.randn(1, n_tokens, d_model) * 0.02)

            layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
            self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

        def forward(self, numeric_tokens, categorical_tokens, *dense_blocks):
            batch_size = numeric_tokens.shape[0] if numeric_tokens.ndim else categorical_tokens.shape[0]
            tokens = [self.cls.expand(batch_size, -1, -1)]
            if self.has_numeric:
                tokens.append(self.numeric_projection(numeric_tokens))
            if self.has_categorical:
                tokens.append(self.categorical_projection(categorical_tokens))
            for projection, block in zip(self.dense_projections, dense_blocks):
                tokens.append(projection(block).unsqueeze(1))
            tokens = torch.cat(tokens, dim=1)
            encoded = self.encoder(tokens + self.position[:, : tokens.shape[1]])
            return self.head(encoded[:, 0]).squeeze(-1)


    class EarlyStopping:
        def __init__(self, patience: int = 2, min_delta: float = 0.0, mode: str = "max"):
            self.patience = patience
            self.min_delta = min_delta
            self.mode = mode
            self.best_score = -math.inf if mode == "max" else math.inf
            self.counter = 0
            self.best_state = None

        def improved(self, score: float) -> bool:
            if self.mode == "max":
                return score > self.best_score + self.min_delta
            return score < self.best_score - self.min_delta

        def step(self, score: float, model: nn.Module) -> bool:
            if self.improved(score):
                self.best_score = score
                self.counter = 0
                self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                return False
            self.counter += 1
            return self.counter >= self.patience

        def restore(self, model: nn.Module) -> None:
            if self.best_state is not None:
                model.load_state_dict(self.best_state)

    print("TinyTokenTransformer is ready.")
else:
    print("torch is not installed in this environment; TinyTokenTransformer cells will skip.")


### セル18: 標準化と、target-free ブロックの一括準備

**何をしているか**
`fit_standardizer` / `apply_standardizer`(平均0分散1に、標準偏差が極小なら1で置換)を定義し、`prepare_target_free_transformer_blocks(frame)` で数値トークン・カテゴリトークン・交互作用・PCA の4ブロックをまとめて作ります。

**なぜそうするのか**
- **なぜ標準偏差が極小のとき1に置き換えるのか**: 定数列を標準化すると `0/0` で NaN や inf になり、**逆伝播で全パラメータが NaN に汚染されます**。1行の防御でモデル全体が守れます。数値計算では「ゼロ割の可能性がある場所を全部潰す」のが基本です。
- **なぜ target-free ブロックだけ先に一括で作るのか(重要な最適化)**: これらは fold に依存しないので、**5 fold で5回作り直す必要がありません**。1回作って使い回せば実行時間が約1/5。一方で target-mean ブロックは fold ごとに作り直さないとリークします。
  - **つまりこのセルの分割そのものが、「リークするもの/しないもの」の境界線を実行効率に翻訳したもの**です。設計が正しいと、性能と正しさが同時に手に入る良い例です。
- **なぜ Transformer には標準化が必要で、XGBoost には要らないのか**: 木は分割点を探すだけなので単調変換に不変です。ニューラルネットは重みの初期化と学習率がスケール前提なので、入力のスケールがそろっていないと学習が不安定になります。


In [ ]:
def fit_standardizer(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = values.mean(axis=0, keepdims=True).astype("float32")
    std = values.std(axis=0, keepdims=True).astype("float32")
    std = np.where(std <= 1e-6, 1.0, std).astype("float32")
    return mean, std


def apply_standardizer(values: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((values - mean) / std).astype("float32")


def prepare_target_free_transformer_blocks(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, list[np.ndarray], dict[str, list[str]]]:
    numeric_tokens, numeric_token_names = build_numeric_token_tensor(frame, state)
    categorical_tokens, categorical_token_names = build_categorical_token_tensor(frame, state)
    interaction_block, interaction_names = build_interaction_features(frame, state)
    pca_source, _ = build_flat_target_free_features(frame, state, include_pca=False)
    pca_block, pca_names = state.transform_pca(pca_source)
    names = {
        "numeric_token_axis": numeric_token_names,
        "categorical_token_axis": categorical_token_names,
        "interaction": interaction_names,
        "pca": pca_names,
    }
    return numeric_tokens, categorical_tokens, [interaction_block, pca_block], names


def predict_tiny_transformer(
    model,
    numeric_tokens: np.ndarray,
    categorical_tokens: np.ndarray,
    dense_blocks: list[np.ndarray],
    batch_size: int,
    device: str,
) -> np.ndarray:
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(numeric_tokens), batch_size):
            stop = start + batch_size
            xb_numeric = torch.from_numpy(numeric_tokens[start:stop]).to(device)
            xb_categorical = torch.from_numpy(categorical_tokens[start:stop]).to(device)
            xb_dense = [torch.from_numpy(block[start:stop]).to(device) for block in dense_blocks]
            logits = model(xb_numeric, xb_categorical, *xb_dense).detach().cpu().numpy()
            preds.append(sigmoid_np(logits))
    return np.concatenate(preds).astype("float32")


### セル19: XGBoost の OOF 予測を Transformer への入力トークンにする(スタッキング)

**何をしているか**
先に学習した XGBoost の結果があれば、その **OOF予測(train用)と test予測**を取り出し、確率そのものと `rank01`(順位を 0〜1 に正規化した値)の2種類にして、Transformer の入力ブロックに追加します。

`rank01(values)` は `pd.Series(values).rank(method='average')` を `(rank - 1) / (n - 1)` で正規化する関数です。

**なぜそうするのか**
- **これはスタッキング(stacking)の一形態です**: 通常のスタッキングは「複数モデルの OOF 予測を入力にして、メタモデルを学習する」ものです。ここでは Transformer 自身がメタモデルを兼ねており、**生の特徴量と XGBoost の意見を同時に見ながら学習します**。
- **なぜ OOF 予測でなければならないのか(絶対条件)**: train 行に対して「その行を学習に使ったモデルの予測」を与えると、その値はほぼ正解そのものです。Transformer は他の特徴を無視してその1本の入力だけを見るようになり、**test では XGBoost の性能を超えられないどころか劣化します**。OOF なら「その行を見ていないモデルの予測」なので、test での状況と同じ条件が再現されます。
- **なぜ確率と順位の両方を渡すのか**: 確率は絶対的な情報(0.9 は 0.6 より強い)、順位は相対的な情報(全体の中で何番目か)。AUC は順位で評価されるので、**順位表現の方が指標に直結**します。両方渡してモデルに選ばせるのが安全です。
- **なぜ `method='average'` なのか**: 同値の扱いを一意にするためです(セル5と同じ理由)。
- **リスク**: XGBoost の OOF にもわずかな楽観バイアスが乗っています。それを入力に使うと**バイアスが増幅されうる**ので、最終ブレンドの重みは必ず OOF 上で探索し、スタックしたモデルを過信しないことが大切です。


In [ ]:
# XGBoost OOF stack token for TinyTokenTransformer

if "_BASE_PREPARE_TARGET_FREE_TRANSFORMER_BLOCKS" not in globals():
    _BASE_PREPARE_TARGET_FREE_TRANSFORMER_BLOCKS = prepare_target_free_transformer_blocks


def rank01(values):
    values = np.asarray(values, dtype="float32")
    ranks = pd.Series(values).rank(method="average").to_numpy(dtype="float32")
    return ((ranks - 1.0) / max(1.0, len(values) - 1.0)).astype("float32")


_XGB_STACK_SOURCE = globals().get("xgboost_stack_result", None)
if _XGB_STACK_SOURCE is None:
    _XGB_STACK_SOURCE = globals().get("xgboost_result", None)

if _XGB_STACK_SOURCE is not None:
    _xgb_train_pred = pd.Series(clip_prob(_XGB_STACK_SOURCE["oof"]), index=train[ID_COL].to_numpy())
    _xgb_test_pred = pd.Series(clip_prob(_XGB_STACK_SOURCE["test_pred"]), index=test[ID_COL].to_numpy())
    _xgb_train_rank = pd.Series(rank01(_XGB_STACK_SOURCE["oof"]), index=train[ID_COL].to_numpy())
    _xgb_test_rank = pd.Series(rank01(_XGB_STACK_SOURCE["test_pred"]), index=test[ID_COL].to_numpy())
    print("TinyTokenTransformer stack source:", _XGB_STACK_SOURCE["name"])
else:
    _xgb_train_pred = _xgb_test_pred = _xgb_train_rank = _xgb_test_rank = None


def xgb_stack_block_for_frame(frame):
    if _XGB_STACK_SOURCE is None:
        return np.zeros((len(frame), 0), dtype="float32"), []

    ids = pd.Series(frame[ID_COL].to_numpy())
    pred = ids.map(_xgb_train_pred).fillna(ids.map(_xgb_test_pred)).fillna(target_prior).to_numpy(dtype="float32")
    rank = ids.map(_xgb_train_rank).fillna(ids.map(_xgb_test_rank)).fillna(0.5).to_numpy(dtype="float32")
    pred = clip_prob(pred)
    logit = np.log(pred / (1.0 - pred)).astype("float32")

    block = np.column_stack([pred, pred - target_prior, logit, rank]).astype("float32")
    names = ["xgb_oof_prob", "xgb_oof_centered", "xgb_oof_logit", "xgb_oof_rank"]
    return block, names


def prepare_target_free_transformer_blocks(frame: pd.DataFrame):
    numeric_tokens, categorical_tokens, dense_blocks, names = _BASE_PREPARE_TARGET_FREE_TRANSFORMER_BLOCKS(frame)
    xgb_block, xgb_names = xgb_stack_block_for_frame(frame)
    if xgb_block.shape[1]:
        dense_blocks = dense_blocks + [xgb_block]
        names = dict(names)
        names["xgb_stack"] = xgb_names
    return numeric_tokens, categorical_tokens, dense_blocks, names


## 10. TinyTokenTransformer Cross Validation

For speed, target-free blocks are built once. The target-mean block is still rebuilt per fold to avoid leakage.


### セル20: TinyTokenTransformer の交差検証

**何をしているか**
1. 乱数シードを `random` / `numpy` / `torch` / `torch.cuda` すべてに設定。
2. デバイスを cuda → mps → cpu の順で選択。
3. **target-free ブロックは train / test それぞれ1回だけ構築**(セル18の最適化)。
4. fold ループ内で、**target-mean ブロックだけ fold の学習部分で fit して再構築**。
5. 各ブロックを fold の学習部分で標準化してから学習。
6. 検証 AUC で早期打ち切りしつつ学習し、OOF と test 予測を集める。

**なぜそうするのか**
- **なぜ4種類ものシードを設定するのか**: Python の `random`、numpy、PyTorch CPU、PyTorch CUDA はそれぞれ**独立した乱数状態**を持ちます。1つ忘れると、dropout やデータシャッフルが実行ごとに変わり、**「変更の効果」と「シードのゆらぎ」が区別できなくなります**。ニューラルネットの実験では再現性の確保が最優先です。
- **なぜデバイス選択を3段にするのか**: Kaggle では GPU、手元の Mac では mps、CI では cpu。**同じコードがどこでも動く**ようにしておくと、デバッグをローカルでできます。
- **なぜ標準化まで fold 内でやるのか**: 平均・標準偏差の計算に目的変数は使わないのでリークはしませんが、**検証部分の情報が学習時の正規化に混ざる**のを避ける保守的な作法です。厳密には transductive でも問題ない場面が多いですが、fold 内に閉じておく方が「CVがLBを予測する」という前提が守られやすくなります。
- **なぜ検証AUCで早期打ち切りするのか**: 最終的に AUC で評価されるので、**打ち切りの基準も AUC にする**のが素直です。logloss で打ち切ると、確率としては良いが順位としては最適でない地点で止まる可能性があります。「**最適化する指標と、選択に使う指標を一致させる**」のは指標駆動の基本です。


In [ ]:
def run_tiny_token_transformer_cv() -> dict[str, object] | None:
    if not RUN_TINY_TOKEN_TRANSFORMER:
        print("RUN_TINY_TOKEN_TRANSFORMER=False; skipping TinyTokenTransformer.")
        return None
    if not TORCH_AVAILABLE:
        print("torch is not installed in this environment; skipping TinyTokenTransformer.")
        return None

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    if torch.cuda.is_available():
        device = "cuda"
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    print("TinyTokenTransformer device:", device)

    train_numeric, train_categorical, train_dense_base, block_names = prepare_target_free_transformer_blocks(train)
    test_numeric, test_categorical, test_dense_base, _ = prepare_target_free_transformer_blocks(test)
    print("numeric tokens:", train_numeric.shape)
    print("categorical tokens:", train_categorical.shape)
    for name, block in zip(["interaction", "pca"], train_dense_base):
        print(f"{name} block:", block.shape)

    folds = make_stratified_folds(y, N_SPLITS, SEED)
    oof = np.zeros(len(train), dtype="float32")
    test_pred = np.zeros(len(test), dtype="float32")
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        fold_start = time.time()
        print(f"\nTinyTokenTransformer fold {fold}/{N_SPLITS}")

        fold_train = train.iloc[tr_idx].reset_index(drop=True)
        fold_target = y[tr_idx]
        train_te_fold, te_names, encoder = build_oof_target_mean_features(
            fold_train,
            fold_target,
            state,
            seed=SEED + 100 + fold,
        )
        valid_te, _ = encoder.transform(train.iloc[va_idx])
        test_te, _ = encoder.transform(test)

        dense_train_raw = [block[tr_idx] for block in train_dense_base] + [train_te_fold]
        dense_valid_raw = [block[va_idx] for block in train_dense_base] + [valid_te]
        dense_test_raw = test_dense_base + [test_te]
        dense_train = []
        dense_valid = []
        dense_test = []
        for block_train_raw, block_valid_raw, block_test_raw in zip(dense_train_raw, dense_valid_raw, dense_test_raw):
            mean, std = fit_standardizer(block_train_raw)
            dense_train.append(apply_standardizer(block_train_raw, mean, std))
            dense_valid.append(apply_standardizer(block_valid_raw, mean, std))
            dense_test.append(apply_standardizer(block_test_raw, mean, std))

        model = TinyTokenTransformer(
            n_numeric_tokens=train_numeric.shape[1],
            numeric_token_dim=train_numeric.shape[2],
            n_categorical_tokens=train_categorical.shape[1],
            categorical_token_dim=train_categorical.shape[2],
            dense_block_dims=[block.shape[1] for block in dense_train],
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=TTT_LR, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, TTT_MAX_EPOCHS),
            eta_min=TTT_LR * 0.05,
        )
        if TTT_USE_POS_WEIGHT:
            pos = max(1.0, float(fold_target.sum()))
            neg = max(1.0, float(len(fold_target) - fold_target.sum()))
            pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        else:
            criterion = nn.BCEWithLogitsLoss()
        early = EarlyStopping(patience=TTT_PATIENCE, mode="max")

        train_dataset = TensorDataset(
            torch.from_numpy(train_numeric[tr_idx]),
            torch.from_numpy(train_categorical[tr_idx]),
            *[torch.from_numpy(block) for block in dense_train],
            torch.from_numpy(y[tr_idx].astype("float32")),
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=TTT_BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            drop_last=False,
            pin_memory=(device == "cuda"),
        )

        for epoch in range(1, TTT_MAX_EPOCHS + 1):
            model.train()
            losses = []
            for batch in train_loader:
                xb_numeric = batch[0].to(device, non_blocking=(device == "cuda"))
                xb_categorical = batch[1].to(device, non_blocking=(device == "cuda"))
                xb_dense = [tensor.to(device, non_blocking=(device == "cuda")) for tensor in batch[2:-1]]
                yb = batch[-1].to(device, non_blocking=(device == "cuda"))

                optimizer.zero_grad(set_to_none=True)
                logits = model(xb_numeric, xb_categorical, *xb_dense)
                loss = criterion(logits, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                losses.append(float(loss.detach().cpu()))

            scheduler.step()
            valid_pred = predict_tiny_transformer(
                model,
                train_numeric[va_idx],
                train_categorical[va_idx],
                dense_valid,
                batch_size=TTT_BATCH_SIZE * 2,
                device=device,
            )
            metrics = prediction_report(y[va_idx], valid_pred)
            print(
                f"epoch={epoch:02d} train_loss={np.mean(losses):.6f} "
                f"valid_auc={metrics['auc']:.6f} valid_logloss={metrics['logloss']:.6f}"
            )
            if early.step(metrics["auc"], model):
                print(f"early stopping at epoch {epoch}; best_auc={early.best_score:.6f}")
                break

        early.restore(model)
        valid_pred = predict_tiny_transformer(
            model,
            train_numeric[va_idx],
            train_categorical[va_idx],
            dense_valid,
            batch_size=TTT_BATCH_SIZE * 2,
            device=device,
        )
        fold_test = predict_tiny_transformer(
            model,
            test_numeric,
            test_categorical,
            dense_test,
            batch_size=TTT_BATCH_SIZE * 2,
            device=device,
        )

        oof[va_idx] = valid_pred
        test_pred += fold_test / N_SPLITS
        metrics = prediction_report(y[va_idx], valid_pred)
        fold_rows.append({"model": "tiny_token_transformer", "fold": fold, **metrics})
        print(
            f"TinyTokenTransformer fold {fold}: auc={metrics['auc']:.6f} "
            f"logloss={metrics['logloss']:.6f} brier={metrics['brier']:.6f} "
            f"runtime={time.time() - fold_start:.1f}s"
        )

        del model, optimizer, scheduler, train_loader, train_dataset, train_te_fold, valid_te, test_te
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_scores = pd.DataFrame(fold_rows)
    full_metrics = prediction_report(y, oof)
    print(f"\nTinyTokenTransformer OOF AUC    : {full_metrics['auc']:.6f}")
    print(f"TinyTokenTransformer OOF logloss: {full_metrics['logloss']:.6f}")
    print(f"TinyTokenTransformer OOF Brier  : {full_metrics['brier']:.6f}")
    display(fold_scores)
    return {"name": "tiny_token_transformer", "oof": clip_prob(oof), "test_pred": clip_prob(test_pred), "fold_scores": fold_scores, "block_names": block_names}


tiny_token_transformer_result = run_tiny_token_transformer_cv()


## 11. TinyTokenTransformer: Seeing the EV Signal

### セル21: TinyTokenTransformer の診断(リフトチャートとキャリブレーション)

**何をしているか**
OOF予測を降順に並べて20ビンに等分し、各ビンの **予測平均**・**実際の正例率**・**リフト**(= ビンの正例率 ÷ 全体の正例率)を計算・可視化します。

**なぜそうするのか**
- **リフトチャートとは**: 「スコア上位10%の層は、全体平均の何倍の確率で買うか」を示す図。**AUC という1つの数字を、実務で使える形に分解したもの**です。
  - AUC 0.945 と言われても現場は動きませんが、「上位10%に絞れば購入率が3.2倍」と言えば販促予算の配分が決まります。**指標をビジネスの言葉に翻訳する**役割です。
- **なぜキャリブレーション(予測平均 vs 実測率)も見るのか**: 理想は45度線上に乗ることです。系統的に上や下にずれていれば、モデルの確率は「順位としては正しいが値としては信用できない」状態です。AUC はそれを検出できないので、**別途見る必要があります**。
- **どこを見るべきか**: 特に**最上位ビンと最下位ビン**です。中間層はどのモデルもそこそこ当たりますが、**「確実に買う人」と「絶対に買わない人」を正しく端に置けるか**がAUCの差を生みます。上位ビンのリフトが伸びていなければ、そこに効く特徴量が足りていないというサインです。


In [ ]:
# TinyTokenTransformer Visualization
def plot_tiny_token_transformer_diagnostics(result, y_true, n_bins=20):
    if result is None:
        print("TinyTokenTransformer was not trained. Check RUN_TINY_TOKEN_TRANSFORMER, torch availability, or earlier errors.")
        return

    if not MPL_AVAILABLE:
        print("matplotlib is not available in this environment.")
        return

    pred = clip_prob(result["oof"])
    y_true = np.asarray(y_true, dtype="float32")

    order = np.argsort(pred)
    sorted_y = y_true[order]
    sorted_pred = pred[order]

    bin_edges = np.linspace(0, len(pred), n_bins + 1, dtype=int)
    rows = []
    for i in range(n_bins):
        start, stop = bin_edges[i], bin_edges[i + 1]
        if stop <= start:
            continue
        rows.append(
            {
                "bin": i + 1,
                "count": stop - start,
                "pred_mean": float(sorted_pred[start:stop].mean()),
                "actual_rate": float(sorted_y[start:stop].mean()),
                "lift": float(sorted_y[start:stop].mean() / max(EPS, y_true.mean())),
            }
        )

    calib = pd.DataFrame(rows)
    metrics = prediction_report(y_true, pred)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    axes[0, 0].hist(pred[y_true == 0], bins=60, alpha=0.65, density=True, label="Actual No", color="#4C78A8")
    axes[0, 0].hist(pred[y_true == 1], bins=60, alpha=0.65, density=True, label="Actual Yes", color="#F58518")
    axes[0, 0].set_title("TinyTokenTransformer OOF Probability Distribution")
    axes[0, 0].set_xlabel("Predicted probability of Will_Buy_EV = Yes")
    axes[0, 0].set_ylabel("Density")
    axes[0, 0].legend()

    axes[0, 1].plot(calib["pred_mean"], calib["actual_rate"], marker="o", color="#54A24B")
    axes[0, 1].plot([0, 1], [0, 1], linestyle="--", color="#888888", linewidth=1)
    axes[0, 1].set_title("Calibration by Probability Bin")
    axes[0, 1].set_xlabel("Mean predicted probability")
    axes[0, 1].set_ylabel("Actual positive rate")
    axes[0, 1].grid(alpha=0.25)

    axes[1, 0].bar(calib["bin"], calib["lift"], color="#E45756")
    axes[1, 0].axhline(1.0, linestyle="--", color="#333333", linewidth=1)
    axes[1, 0].set_title("Lift by Sorted Prediction Bin")
    axes[1, 0].set_xlabel("Prediction bin, low to high")
    axes[1, 0].set_ylabel("Lift vs global positive rate")

    cumulative_positive = np.cumsum(sorted_y[::-1])
    population_share = np.arange(1, len(y_true) + 1) / len(y_true)
    positive_capture = cumulative_positive / max(1.0, y_true.sum())

    axes[1, 1].plot(population_share, positive_capture, color="#B279A2", linewidth=2)
    axes[1, 1].plot([0, 1], [0, 1], linestyle="--", color="#888888", linewidth=1)
    axes[1, 1].set_title("Cumulative Gain Curve")
    axes[1, 1].set_xlabel("Top share of rows by predicted probability")
    axes[1, 1].set_ylabel("Captured share of actual Yes")
    axes[1, 1].grid(alpha=0.25)

    fig.suptitle(
        f"TinyTokenTransformer Diagnostics | AUC={metrics['auc']:.5f} "
        f"Logloss={metrics['logloss']:.5f} Brier={metrics['brier']:.5f}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()

    display(calib)


plot_tiny_token_transformer_diagnostics(tiny_token_transformer_result, y)

## 12. OOF Blend and Calibration

The ensemble searches non-negative weights for:

- the global target prior
- XGBoost OOF probabilities
- TinyTokenTransformer OOF probabilities

AUC chooses the ranker; logloss breaks ties so the selected blend still behaves like a usable probability.


### セル22: OOF 貪欲ブレンド(V7:多ビュー版)

**何をしているか**
すべてのモデル結果(XGBoost、XGBoostのバリアント群、スタック版、TinyTokenTransformer)について、**3つの「見方」**を候補に展開します。

- `prob` — 確率そのまま
- `rank01` — 予測値の順位を 0〜1 に正規化
- `logit_rank01` — 確率をロジット変換してから順位を取り 0〜1 に正規化

そのうえで**貪欲な前向き選択(greedy forward blending)**を回します:

```
現在のブレンド ← 最良の単体モデル
繰り返し:
    すべての候補 c、すべての混合比 α について
        試行 = (1-α) * 現在 + α * c
        OOF AUC を計算(同点なら logloss で比較)
    最良の (c, α) を採用。AUC が改善しなければ終了
```

**なぜそうするのか**
- **なぜ順位に直してから混ぜるのか(AUC最適化の核心)**: 2つのモデルの確率のスケールが違う場合(片方が 0.1〜0.9、もう片方が 0.4〜0.6 に集中)、単純平均すると**レンジの広いモデルが支配してしまいます**。順位に直せば両方とも一様に 0〜1 に広がるので、対等に足せます。そして AUC は順位しか見ないので、**順位空間で混ぜても情報は一切失われません**。これは AUC 指標のコンペで最も費用対効果の高いテクニックの1つです。
- **なぜ `logit_rank01` も用意するのか**: ロジット変換は両端を引き伸ばすので、**確率が0や1に近い領域の細かい差**が順位に反映されやすくなります。確率がぎゅっと詰まったモデルでは、生の確率で順位を取ると同値が増えて情報が潰れます。
- **なぜ貪欲法なのか**: 最適な重みベクトルを一発で解こうとすると、候補数だけ次元があり、**OOF に対して過学習します**。貪欲な前向き選択は「1回に1つだけ、しかも改善するときだけ追加する」ので暗黙の正則化がかかり、少数のモデルに重みが集中しやすい。Kaggle で hill climbing と呼ばれる定番手法です。
- **なぜ「改善しなければ break」なのか**: 過学習の歯止めです。`best_auc + 1e-12` という微小マージンを置いて、**数値誤差レベルの"改善"では続行しない**ようにしています。
- **なぜ logloss をタイブレークにするのか**: AUC は離散的な順位の指標なので、候補が違っても同じ値になることがあります。そのとき logloss で選べば、**同じ順位性能なら確率としてまともな方**が選ばれます。順位を主・確率を従とする、指標に忠実な優先順位です。
- **注意点**: この探索は OOF 上で行われるので、**候補数を増やすほど OOF に過学習します**。ブレンド後のOOF AUC は、実際のLBよりわずかに楽観的だと考えておくべきです。


In [ ]:
# V7 generalized greedy blend for XGBoost-family views and TinyTokenTransformer

def blend_rank01(values):
    values = np.asarray(values, dtype="float32")
    ranks = pd.Series(values).rank(method="average").to_numpy(dtype="float32")
    return clip_prob((ranks - 1.0) / max(1.0, len(values) - 1.0))


def blend_logit_rank01(values):
    values = clip_prob(values)
    logits = np.log(values / (1.0 - values)).astype("float32")
    ranks = pd.Series(logits).rank(method="average").to_numpy(dtype="float32")
    return clip_prob((ranks - 1.0) / max(1.0, len(values) - 1.0))


model_results = [
    result
    for result in [
        globals().get("xgboost_result"),
        *globals().get("xgboost_variant_results", []),
        globals().get("xgboost_stack_result"),
        globals().get("tiny_token_transformer_result"),
    ]
    if result is not None
]

if not model_results:
    raise RuntimeError("No XGBoost or TinyTokenTransformer results are available for blending.")

candidate_rows = []
seen = set()
for result in model_results:
    for view, transform in [
        ("prob", clip_prob),
        ("rank", blend_rank01),
        ("logit_rank", blend_logit_rank01),
    ]:
        name = f"{result['name']}__{view}"
        if name in seen:
            continue
        seen.add(name)
        candidate_rows.append(
            {
                "name": name,
                "model": result["name"],
                "view": view,
                "oof": transform(result["oof"]),
                "test": transform(result["test_pred"]),
            }
        )


def _blend_score(pred: np.ndarray) -> tuple[float, float]:
    metrics = prediction_report(y, pred)
    return metrics["auc"], -metrics["logloss"]


single_rows = []
for candidate in candidate_rows:
    metrics = prediction_report(y, candidate["oof"])
    single_rows.append({"candidate": candidate["name"], **metrics})
candidate_score_df = pd.DataFrame(single_rows).sort_values(["auc", "logloss"], ascending=[False, True])
display(candidate_score_df)

best_candidate = max(candidate_rows, key=lambda item: _blend_score(item["oof"]))
current_oof = best_candidate["oof"].copy()
current_test = best_candidate["test"].copy()
blend_recipe = [(best_candidate["name"], 1.0)]
best_auc, best_neg_logloss = _blend_score(current_oof)

alpha_grid = np.array([0.01, 0.02, 0.03, 0.05, 0.08, 0.13, 0.21, 0.34, 0.50], dtype="float32")
MAX_BLEND_ROUNDS = 80

for step in range(MAX_BLEND_ROUNDS):
    step_best = None
    for candidate in candidate_rows:
        for alpha in alpha_grid:
            trial_oof = clip_prob((1.0 - alpha) * current_oof + alpha * candidate["oof"])
            auc, neg_logloss = _blend_score(trial_oof)
            if step_best is None or auc > step_best["auc"] + 1e-12 or (
                abs(auc - step_best["auc"]) <= 1e-12 and neg_logloss > step_best["neg_logloss"]
            ):
                step_best = {
                    "candidate": candidate,
                    "alpha": float(alpha),
                    "auc": auc,
                    "neg_logloss": neg_logloss,
                    "oof": trial_oof,
                    "test": clip_prob((1.0 - alpha) * current_test + alpha * candidate["test"]),
                }

    if step_best is None or step_best["auc"] <= best_auc + 1e-12:
        break

    alpha = step_best["alpha"]
    blend_recipe = [(name, weight * (1.0 - alpha)) for name, weight in blend_recipe]
    blend_recipe.append((step_best["candidate"]["name"], alpha))
    current_oof = step_best["oof"]
    current_test = step_best["test"]
    best_auc = step_best["auc"]
    best_neg_logloss = step_best["neg_logloss"]
    print(f"blend step {step + 1:02d}: +{step_best['candidate']['name']} alpha={alpha:.2f} auc={best_auc:.8f}")

recipe_series = pd.Series(dtype="float64")
for name, weight in blend_recipe:
    recipe_series.loc[name] = recipe_series.get(name, 0.0) + weight
recipe_series = recipe_series.sort_values(ascending=False)

blend_weights = {str(name): float(weight) for name, weight in recipe_series.items()}
blend_view = "xgb_ttt_greedy_multi_view"
blended_oof = clip_prob(current_oof).astype("float32")
final_test_pred = clip_prob(current_test).astype("float32")
blend_metrics = prediction_report(y, blended_oof)

for result in model_results:
    metrics = prediction_report(y, result["oof"])
    print(f"{result['name']} OOF AUC={metrics['auc']:.6f} logloss={metrics['logloss']:.6f}")

print("Blend view:", blend_view)
print("Blend recipe:")
display(recipe_series.rename("weight").reset_index().rename(columns={"index": "candidate"}))
print(f"Blended OOF AUC    : {blend_metrics['auc']:.6f}")
print(f"Blended OOF logloss: {blend_metrics['logloss']:.6f}")


### セル23: キャリブレーション表とモデル比較

**何をしているか**
各モデルとブレンドについて、予測を12ビンに分けた **予測平均 vs 実測率** の表を作り、AUC / logloss / Brier を並べた比較表を AUC 降順で表示。キャリブレーション曲線も重ねて描きます。

**なぜそうするのか**
- **なぜ全モデルを並べて比較するのか**: ブレンドが単体モデルより本当に良いのかを確認するためです。**貪欲探索が OOF に過学習していると、ブレンドの OOF AUC は上がるのに LB では単体モデルに負ける**ことがあります。差が極めて小さい(例:+0.00003)なら、ブレンドではなく単体を選ぶ判断もありえます。
- **なぜ複数モデルのキャリブレーション曲線を重ねるのか**: どのモデルがどの確率帯で信頼できるかが見えます。XGBoost は中間帯で素直、Transformer は端が過信気味、といった傾向が読み取れれば、**ブレンド重みの妥当性の裏取り**になります。
- **一般則**: 「OOF AUC が最も高いものを選ぶ」だけでは足りません。**改善幅が fold 間分散に対して十分大きいか**を必ず確認してください。fold ごとの AUC の標準偏差が 0.001 なのに改善が 0.0001 なら、それは誤差です。


In [ ]:
def calibration_table(y_true, pred, n_bins=12):
    pred = clip_prob(pred)
    order = np.argsort(pred)
    y_sorted = y_true[order]
    p_sorted = pred[order]
    edges = np.linspace(0, len(pred), n_bins + 1, dtype=int)

    rows = []
    for i in range(n_bins):
        s, e = edges[i], edges[i + 1]
        rows.append({
            "bin": i + 1,
            "pred_mean": float(p_sorted[s:e].mean()),
            "actual_rate": float(y_sorted[s:e].mean()),
            "count": int(e - s),
        })
    return pd.DataFrame(rows)

plot_items = []
for result in model_results:
    plot_items.append((result["name"], result["oof"]))
plot_items.append(("blend", blended_oof))

score_rows = []
for name, pred in plot_items:
    m = prediction_report(y, pred)
    score_rows.append({"model": name, **m})

score_df = pd.DataFrame(score_rows).sort_values("auc", ascending=False)
display(score_df)

if MPL_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for name, pred in plot_items:
        tab = calibration_table(y, pred)
        axes[0].plot(tab["pred_mean"], tab["actual_rate"], marker="o", label=name)

    axes[0].plot([0, 1], [0, 1], "--", color="#888888", linewidth=1)
    axes[0].set_title("OOF Calibration")
    axes[0].set_xlabel("mean predicted probability")
    axes[0].set_ylabel("actual positive rate")
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    axes[1].bar(score_df["model"], score_df["auc"], color="#4C78A8")
    axes[1].set_title("OOF AUC Comparison")
    axes[1].set_ylabel("AUC")
    axes[1].set_ylim(max(0.5, score_df["auc"].min() - 0.02), min(1.0, score_df["auc"].max() + 0.02))

    plt.tight_layout()
    plt.show()

## 13. Final Submission

The test prediction uses the same weights found on OOF predictions and writes positive-class probabilities to `submission.csv`.


### セル24: 最終提出ファイルの書き出しと実行サマリの保存

**何をしているか**
- OOF で見つけた重みをそのまま test 予測に適用し、`submission.csv` を書く。
- **`oof_predictions.csv`** に、正解・ブレンド予測・各モデルのOOF予測を全部保存。
- **`fold_scores.csv`** に fold ごとのスコアを保存。
- **`run_summary`** に、目的変数名、正例ラベル、train/test の形、数値列・カテゴリ列のリスト、学習したモデル名、ブレンド重み、定数予測のAUC(=0.5のベースライン)などを JSON で保存。

**なぜそうするのか**
- **なぜ OOF で見つけた重みをそのまま使うのか**: test にはラベルが無いので、重みを test 上で調整することは原理的にできません(できるならそれはLB過学習です)。**OOF は「未見データに対する予測」なので、test での挙動の最良の代理**です。
- **なぜ OOF 予測を CSV に残すのか(実務で最も効く一手)**: 明日別の notebook を書いたとき、この OOF ファイルがあれば**再学習なしにブレンド相手として使えます**。Kaggleの上位陣は例外なく OOF を資産として蓄積しています。1日1本ずつ OOF を貯めれば、月末には30本のアンサンブル素材になります。
- **なぜ定数予測の AUC を記録するのか**: 全員に同じ確率を出したときの AUC は定義上 0.5 です。これが**サニティチェック**になります。もし 0.5 から大きくずれていたら、AUC の実装かラベルの向きが壊れています。**「絶対にこうなるはずの値」を1つ計算して確認する**のは、自前実装をした人間の義務です。
- **なぜ実行サマリを JSON で残すのか**: 3日後に「この submission はどの設定で作ったか」を思い出すためです。ファイル名だけでは足りません。**成果物と設定を必ずセットで保存する**。


In [ ]:
test_blend = clip_prob(final_test_pred)

submission = sample_submission.copy()
submission[TARGET] = test_blend
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

oof_frame = pd.DataFrame({ID_COL: train[ID_COL], TARGET: y, "blend_pred": blended_oof})
for result in model_results:
    oof_frame[f"{result['name']}_pred"] = result["oof"]
oof_frame.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)

score_frames = [result["fold_scores"] for result in model_results if "fold_scores" in result]
if score_frames:
    pd.concat(score_frames, ignore_index=True).to_csv(OUTPUT_DIR / "fold_scores.csv", index=False)

run_summary = {
    "target": TARGET,
    "positive_label": POSITIVE_LABEL,
    "target_mapping": TARGET_MAPPING,
    "train_shape": list(train.shape),
    "test_shape": list(test.shape),
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "models_trained": [result["name"] for result in model_results],
    "blend_weights": blend_weights,
    "constant_baseline_auc": roc_auc_score_np(y, constant_pred),
    "constant_baseline_logloss": binary_log_loss(y, constant_pred),
    "blended_oof_auc": blend_metrics["auc"],
    "blended_oof_logloss": blend_metrics["logloss"],
    "blended_oof_brier": blend_metrics["brier"],
    "submission_path": str(submission_path),
}
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")

display(submission.head())
print("Saved:", submission_path)


## Closing Notes

This is now the S6E9 classification version of the two-trunk token notebook:

- XGBoost tests whether the engineered token matrix contains split-friendly signal.
- TinyTokenTransformer tests whether numeric and categorical field tokens can learn useful attention patterns.
- Target-mean tokens are leakage-safe through nested OOF construction.
- The final blend uses OOF AUC first and logloss as the calibration tie-breaker.